# 01 — Raw Collection | Full Scraping Bertahap (Production-Safe)

Penelitian sentimen temporal tentang **keputusan penetapan/tidak ditetapkannya status bencana nasional** di X/Twitter: Lombok 2018, Palu 2018, dan Sumatra 2025.

**KONDISI AWAL: PREVIEW-ONLY.** Tidak ada panggilan berbayar saat menjalankan semua cell dengan konfigurasi bawaan. Jangan mengaktifkan `EXECUTE_SCRAPING` sebelum mengonfirmasi saldo tersedia, memahami risiko biaya, dan melihat preview serta hasil tes.

Versi ini mempertahankan `CONFIG_VERSION`, `ACTOR_ID`, `task_key()`, struktur folder dan checkpoint dari pipeline pilot. Hasil pilot tetap dibaca dan tidak dihapus. **Simpan notebook di `01_Raw Data/namafile.ipynb`

**Prinsip:** `E01_P01` satu query awal per kasus, jendela 6 jam, round-robin, hanya sedikit Actor calls per batch, batas USD per run dan per sesi, checkpoint append-only, serta penghitungan ID unik global. `MAX_ITEMS` input Actor bukan bukti kelengkapan.

> Catatan anggaran: pengguna melaporkan **$1,21 sudah digunakan**. Itu bukan saldo tersisa. Nilai `AVAILABLE_CREDIT_USD` harus berasal dari dashboard Apify **saat batch akan dijalankan**; tidak boleh dihitung dengan mengurangkan asumsi tertentu.

**Pembaharuan v2:** prioritas fase keputusan tanpa menghapus tanggal, pause 3 run berturut-turut tanpa tweet asli, status data terpisah dari SAVED, rekonsiliasi usage $1.21→$1.22, dan laporan global lintas pilot/full.


In [62]:
# 01 — IMPORT & VERSI
from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import time
from collections import Counter, defaultdict
from datetime import datetime, timedelta, timezone
from decimal import Decimal, InvalidOperation
from getpass import getpass
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import pandas as pd
from IPython.display import display

try:
    from apify_client import ApifyClient
except ImportError as exc:
    raise ImportError(
        "Instal apify-client dan pandas pada kernel VS Code yang dipakai. "
        "Rekomendasi: uv pip install apify-client pandas"
    ) from exc

print('Pandas:', pd.__version__)
print('Import berhasil (belum ada panggilan API).')


Pandas: 3.0.6
Import berhasil (belum ada panggilan API).


In [63]:
# 02 — LOKASI PROJECT & OUTPUT
# Repository memiliki folder bernama "01_Raw Data" (spasi), bukan "01_Raw_Data".
# Jika Jupyter dibuka dari lokasi lain, set PROJECT_ROOT melalui environment variable.
root_env = os.getenv('PROJECT_ROOT')
if root_env:
    PROJECT_ROOT = Path(root_env).expanduser().resolve()
elif (Path.cwd() / '01_Raw Data').is_dir():
    PROJECT_ROOT = Path.cwd().resolve()
elif Path.cwd().name == '01_Raw Data':
    PROJECT_ROOT = Path.cwd().parent.resolve()
else:
    raise RuntimeError(
        'Folder 01_Raw Data tidak ditemukan. Buka notebook dari repository atau set PROJECT_ROOT.'
    )

STAGE_DIR = PROJECT_ROOT / '01_Raw Data'
OUT_DIR = STAGE_DIR / 'outputs'
RAW_DIR = OUT_DIR / 'raw_jsonl'
CSV_DIR = OUT_DIR / 'raw_csv'
AUDIT_DIR = OUT_DIR / 'audit'
STATE_DIR = OUT_DIR / 'state'
REPORT_DIR = OUT_DIR / 'reports'
for folder in [RAW_DIR, CSV_DIR, AUDIT_DIR, STATE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

CHECKPOINT_FILE = STATE_DIR / 'collection_checkpoint.jsonl'
CONFIG_VERSION = 'policy_v2_2026_10'
ACTOR_ID = 'CJdippxWmn9uRfooo'
WIB = ZoneInfo('Asia/Jakarta')
UTC = timezone.utc

print('Project:', PROJECT_ROOT)
print('Output :', OUT_DIR)
print('Actor  :', ACTOR_ID)

Project: C:\Penelitian Sentimen Bencana Nasional
Output : C:\Penelitian Sentimen Bencana Nasional\01_Raw Data\outputs
Actor  : CJdippxWmn9uRfooo


## Langkah 1 — Konfigurasi kasus, kata kunci, dan query
Ketiga periode **inklusif**. Semua tanggal tweet dan jendela pengambilan dihitung dalam WIB.
Kelompok EVENT harus berpasangan dengan kelompok POLICY. Tidak ada scraping event-only.
Keyword luas (contohnya "status darurat", "pemerintah pusat") ditandai untuk peninjauan,
bukan otomatis dianggap sebagai opini tentang status bencana nasional.

In [19]:
# 03 — KONFIGURASI PENELITIAN (dipertahankan dari notebook semula)
DISASTER_CASES = {
    'lombok_2018': {
        'label': 'Gempa Lombok 2018',
        'start_date': '2018-07-29', 'end_date': '2018-08-31',
        'pilot_date': '2018-08-06',
        'event_terms': [
            'Lombok', 'gempa Lombok', 'gempa bumi Lombok', 'gempa NTB',
            'gempa bumi NTB', 'NTB', 'Nusa Tenggara Barat', 'Mataram',
            'Lombok Utara', 'Lombok Barat', 'Lombok Timur', 'Lombok Tengah'
        ],
        'policy_terms': [
            'bencana nasional', 'status bencana nasional', 'status nasional',
            'penetapan bencana nasional', 'penetapan status bencana',
            'penetapan status', 'status bencana', 'status darurat',
            'bencana daerah', 'status bencana daerah',
            'ditetapkan sebagai bencana nasional', 'ditetapkan bencana nasional',
            'tetapkan bencana nasional', 'menetapkan bencana nasional',
            'jadi bencana nasional', 'jadiin bencana nasional'
        ],
    },
    'palu_2018': {
        'label': 'Gempa dan Tsunami Palu 2018',
        'start_date': '2018-09-28', 'end_date': '2018-10-31',
        'pilot_date': '2018-09-29',
        'event_terms': [
            'Palu', 'gempa Palu', 'gempa bumi Palu', 'tsunami Palu',
            'gempa tsunami Palu', 'Donggala', 'gempa Donggala',
            'tsunami Donggala', 'Sulawesi Tengah', 'gempa Sulawesi Tengah',
            'tsunami Sulawesi Tengah', 'bencana Palu', 'bencana Sulawesi Tengah'
        ],
        'policy_terms': [
            'bencana nasional', 'status bencana nasional', 'status nasional',
            'penetapan bencana nasional', 'penetapan status bencana',
            'penetapan status', 'status bencana', 'status darurat',
            'bencana daerah', 'status bencana daerah',
            'ditetapkan sebagai bencana nasional', 'ditetapkan bencana nasional',
            'tetapkan bencana nasional', 'menetapkan bencana nasional',
            'jadi bencana nasional', 'bantuan internasional'
        ],
    },
    'sumatra_2025': {
        'label': 'Banjir dan Longsor Sumatra 2025',
        'start_date': '2025-11-26', 'end_date': '2025-12-31',
        'pilot_date': '2025-11-29',
        'event_terms': [
            'Aceh', 'banjir Aceh', 'longsor Aceh', 'bencana Aceh',
            'Sumatra Utara', 'Sumatera Utara', 'banjir Sumatra Utara',
            'banjir Sumatera Utara', 'longsor Sumatra Utara',
            'longsor Sumatera Utara', 'bencana Sumatra Utara',
            'bencana Sumatera Utara', 'Sumatra Barat', 'Sumatera Barat',
            'banjir Sumatra Barat', 'banjir Sumatera Barat',
            'longsor Sumatra Barat', 'longsor Sumatera Barat',
            'bencana Sumatra Barat', 'bencana Sumatera Barat',
            'banjir Sumatra', 'banjir Sumatera', 'longsor Sumatra',
            'longsor Sumatera', 'bencana Sumatra', 'bencana Sumatera'
        ],
        'policy_terms': [
            'bencana nasional', 'status bencana nasional', 'status nasional',
            'penetapan bencana nasional', 'penetapan status bencana',
            'penetapan status', 'status bencana', 'status darurat',
            'bencana daerah', 'status bencana daerah',
            'ditetapkan sebagai bencana nasional', 'ditetapkan bencana nasional',
            'tetapkan bencana nasional', 'menetapkan bencana nasional',
            'jadi bencana nasional', 'prioritas nasional',
            'penanganan nasional', 'penanganan berskala nasional',
            'berskala nasional', 'ditangani pemerintah pusat',
            'penanganan pemerintah pusat', 'pemerintah pusat',
            'status daerah', 'status provinsi'
        ],
    },
}

STRONG_POLICY_TERMS = [
    'bencana nasional', 'status bencana nasional',
    'penetapan bencana nasional', 'ditetapkan sebagai bencana nasional',
    'ditetapkan bencana nasional', 'tetapkan bencana nasional',
    'menetapkan bencana nasional', 'jadi bencana nasional',
    'jadiin bencana nasional', 'status nasional'
]

BROAD_POLICY_TERMS = [
    'penetapan status bencana', 'penetapan status', 'status bencana',
    'status darurat', 'bencana daerah', 'status bencana daerah',
    'bantuan internasional', 'prioritas nasional', 'penanganan nasional',
    'penanganan berskala nasional', 'berskala nasional',
    'ditangani pemerintah pusat', 'penanganan pemerintah pusat',
    'pemerintah pusat', 'status daerah', 'status provinsi'
]

# Grup query yang sama seperti notebook lama (5 event × 6 policy / batch).
EVENT_CHUNK_SIZE = 5
POLICY_CHUNK_SIZE = 6


def chunks(items: list[str], size: int):
    if size <= 0:
        raise ValueError('Ukuran batch harus > 0')
    for i in range(0, len(items), size):
        yield items[i:i + size]


def or_query(terms: list[str]) -> str:
    return '(' + ' OR '.join('"' + term.replace('"', '') + '"' for term in terms) + ')'


def make_query_batches(case_config: dict) -> list[dict]:
    batches = []
    for e_index, events in enumerate(chunks(case_config['event_terms'], EVENT_CHUNK_SIZE), 1):
        for p_index, policies in enumerate(chunks(case_config['policy_terms'], POLICY_CHUNK_SIZE), 1):
            # X search: spasi antarkelompok bermakna konjungsi/AND.
            # Keyword kebijakan tetap diwajibkan; hindari literal 'AND' dalam kueri.
            base = f'{or_query(events)} {or_query(policies)}'
            batches.append({
                'query_id': f'E{e_index:02d}_P{p_index:02d}',
                'base_query': base,
                'event_terms': events,
                'policy_terms': policies,
            })
    return batches

for case_id, conf in DISASTER_CASES.items():
    conf['query_batches'] = make_query_batches(conf)
    print(case_id, '| event:', len(conf['event_terms']),
          '| policy:', len(conf['policy_terms']),
          '| query batch:', len(conf['query_batches']))

assert [len(DISASTER_CASES[c]['query_batches']) for c in DISASTER_CASES] == [9, 9, 24]

lombok_2018 | event: 12 | policy: 16 | query batch: 9
palu_2018 | event: 13 | policy: 16 | query batch: 9
sumatra_2025 | event: 26 | policy: 24 | query batch: 24


## Langkah 2 — Pilih strategi produksi tanpa API call

- **Selalu mulai dengan `EXECUTE_SCRAPING=False`**; `RUN_MODE="full"` saja tidak memanggil API.
- **Jangan mengganti `CONFIG_VERSION`** kecuali memang ingin membuat checkpoint baru untuk query berbeda. Format `task_key` dipertahankan agar task pilot 6 jam dengan query sama tidak terbayar ulang.
- `FULL_WINDOW_HOURS=6` dan `FULL_QUERY_IDS` hanya memilih `E01_P01`. Ini **belum** mencakup seluruh query atau seluruh opini; query alternatif dapat ditambahkan pada fase berikutnya setelah melihat yield dan cakupan temporal.
- `FULL_HOUR_PRIORITY=(6, 12, 18, 0)` memprioritaskan jam yang pada sampel pilot lebih produktif; **seluruh empat jendela harian tetap direncanakan**, sehingga dataset akhir tidak boleh dianalisis sebelum cakupan hari lengkap.
- Target **4.000–5.000 ID tweet asli unik *relevan*** adalah sasaran, bukan jaminan. Pemberitaan, promosi, dan tweet di luar kebijakan perlu divalidasi semantik.
- `max_items` dan `max_total_charge_usd` adalah **parameter tingkat run pada Apify API**, bukan sekadar properti input Actor. Tetap cek harga Actor serta biaya aktual dashboard.


In [64]:
# 04 — KONFIGURASI FULL COLLECTION BERTAHAP (DEFAULT: TIDAK BERBAYAR)
RUN_MODE = 'full'
EXECUTE_SCRAPING = False
PAID_RUN_CONFIRMATION = ''  # Hanya isi jika memang siap: I_APPROVE_PAID_FULL_SCRAPING

# Pembaruan dashboard sesudah smoke test: $1.22 terpakai dari batas $5.
CREDIT_USED_TO_DATE_USD = Decimal('1.28')  # Informasi; tidak dipakai menghitung sisa saldo.
AVAILABLE_CREDIT_USD = Decimal('3.72')               # WAJIB isi sesuai sisa kredit dashboard Apify sebelum PAID.
CREDIT_RESERVE_USD = Decimal('0.50')       # Jangan gunakan cadangan ini.
MAX_SESSION_SPEND_USD = Decimal('0.12')    # Batas komitmen maksimal SATU sesi notebook.
MAX_CHARGE_PER_RUN_USD = Decimal('0.03')   # Apify max_total_charge_usd per run.
MAX_NEW_RUNS = 3                           # Batch kecil, maksimal 3 panggilan baru.
MAX_CONSECUTIVE_NO_REAL = 3                # Pause sesi setelah 3 run baru berturut-turut 0 tweet asli.
MAX_ITEMS = 20                             # Hint Actor + server cap max_items (bukan batas raw pasti).
RETRY_UNCERTAIN_TASKS = False              # Jangan bayar ulang task berstatus NEEDS_REVIEW.

# Pengumpulan lintas kasus: satu query pertama per kasus, seluruh tanggal, jendela 6 jam.
FULL_CASES = ('lombok_2018', 'palu_2018', 'sumatra_2025')
FULL_QUERY_IDS = {
    'lombok_2018': ('E01_P01',),
    'palu_2018': ('E01_P01',),
    'sumatra_2025': ('E01_P01',),
}
FULL_WINDOW_HOURS = 6
FULL_HOUR_PRIORITY = (6, 12, 18, 0)  # Semua jam masih dikunjungi; prioritas saja.
# Tanggal berpotensi ramai DIDAHULUKAN saja; tanggal lain tetap masuk rencana FULL.
FULL_PRIORITY_DATES = {
    'lombok_2018': ('2018-08-14', '2018-08-13'),
    'palu_2018': ('2018-10-03', '2018-10-02'),
    'sumatra_2025': ('2025-12-04', '2025-12-03'),
}
FULL_MAX_DAYS_PER_CASE = None  # Mis. 2 untuk uji rencana, bukan rekomendasi dataset final.

# Tetap menyediakan mode pilot untuk regresi logika; full mode yang digunakan saat produksi.
PILOT_QUERY_LIMIT = 1
PILOT_HOURS = 6
PILOT_START_HOUR = 6
ENABLE_ADAPTIVE = False                 # Default OFF karena dapat membuat panggilan tambahan.
MAX_ADAPTIVE_DEPTH = 3
REQUEST_DELAY_SECONDS = 1.0

assert RUN_MODE in {'pilot', 'full'}
assert 1 <= PILOT_HOURS <= 24 and 0 <= PILOT_START_HOUR <= 23
assert PILOT_START_HOUR + PILOT_HOURS <= 24
assert FULL_WINDOW_HOURS in {1, 3, 6, 12, 24}
assert MAX_NEW_RUNS >= 1 and MAX_ITEMS >= 1
assert MAX_CONSECUTIVE_NO_REAL >= 2
for _c, _days in FULL_PRIORITY_DATES.items():
    assert _c in DISASTER_CASES and len(set(_days)) == len(_days)
    assert all(DISASTER_CASES[_c]['start_date'] <= d <= DISASTER_CASES[_c]['end_date'] for d in _days)
assert MAX_SESSION_SPEND_USD > 0 and MAX_CHARGE_PER_RUN_USD > 0
assert MAX_SESSION_SPEND_USD >= MAX_CHARGE_PER_RUN_USD
assert set(FULL_CASES).issubset(DISASTER_CASES)
assert len(FULL_HOUR_PRIORITY) == 24 // FULL_WINDOW_HOURS
assert sorted(FULL_HOUR_PRIORITY) == list(range(0, 24, FULL_WINDOW_HOURS))
assert set(FULL_QUERY_IDS) == set(FULL_CASES)
assert not RETRY_UNCERTAIN_TASKS, 'Retry task ambigu harus dilakukan manual, bukan otomatis.'

for _case_id in FULL_CASES:
    _available_query_ids = {b['query_id'] for b in DISASTER_CASES[_case_id]['query_batches']}
    assert set(FULL_QUERY_IDS[_case_id]).issubset(_available_query_ids)


def wib_dt(day: str, hour: int = 0) -> datetime:
    return datetime.fromisoformat(day).replace(hour=hour, tzinfo=WIB)


def iso_wib(dt: datetime) -> str:
    return dt.astimezone(WIB).strftime('%Y-%m-%d %H:%M:%S%z')


def make_search_query(base_query: str, start_wib: datetime, end_wib: datetime) -> str:
    start_unix = int(start_wib.timestamp())
    end_unix = int(end_wib.timestamp())
    return f'{base_query} lang:in since_time:{start_unix} until_time:{end_unix}'


# IDENTITAS TASK DARI PILOT DIPERTAHANKAN — JANGAN UBAH TANPA ALASAN.
def task_key(task: dict) -> str:
    identifying = {
        'case': task['case'], 'base_query': task['base_query'],
        'start_wib': iso_wib(task['start_wib']),
        'end_wib': iso_wib(task['end_wib']),
        'config_version': CONFIG_VERSION, 'actor_id': ACTOR_ID,
    }
    digest = hashlib.sha256(
        json.dumps(identifying, sort_keys=True, ensure_ascii=False).encode('utf-8')
    ).hexdigest()[:20]
    return f"{task['case']}_{digest}"


def task_info(case_id: str, batch: dict, start: datetime, end: datetime, depth=0) -> dict:
    task = {
        'case': case_id, 'query_id': batch['query_id'],
        'base_query': batch['base_query'],
        'start_wib': start, 'end_wib': end, 'depth': depth,
    }
    task['task_id'] = task_key(task)
    task['query'] = make_search_query(task['base_query'], start, end)
    return task


def date_windows(conf: dict, hours: int):
    start = wib_dt(conf['start_date'])
    end_exclusive = wib_dt(conf['end_date']) + timedelta(days=1)
    step = timedelta(hours=hours)
    while start < end_exclusive:
        stop = min(start + step, end_exclusive)
        yield start, stop
        start = stop


def build_tasks(mode: str = RUN_MODE) -> list[dict]:
    # Round-robin antar kasus dan hari. Mode full tidak memproses 42 batch query.
    case_tasks = {}
    for case_id, conf in DISASTER_CASES.items():
        group = []
        if mode == 'pilot':
            start = wib_dt(conf['pilot_date'], PILOT_START_HOUR)
            windows = [(start, start + timedelta(hours=PILOT_HOURS))]
            batches = conf['query_batches'][:PILOT_QUERY_LIMIT]
        elif mode == 'full':
            if case_id not in FULL_CASES:
                case_tasks[case_id] = []
                continue
            grouped = defaultdict(list)
            for start, end in date_windows(conf, FULL_WINDOW_HOURS):
                grouped[start.date()].append((start, end))
            dates = sorted(grouped.keys())
            # Prioritas tanggal mengubah urutan, BUKAN membuang hari.
            preferred = [datetime.fromisoformat(d).date()
                        for d in FULL_PRIORITY_DATES.get(case_id, ())]
            dates = [d for d in preferred if d in grouped] + [
                d for d in dates if d not in preferred]
            if FULL_MAX_DAYS_PER_CASE is not None:
                dates = dates[:FULL_MAX_DAYS_PER_CASE]
            windows = []
            for date in dates:
                per_day = sorted(
                    grouped[date],
                    key=lambda pair: FULL_HOUR_PRIORITY.index(pair[0].hour),
                )
                windows.extend(per_day)
            batch_by_id = {b['query_id']: b for b in conf['query_batches']}
            batches = [batch_by_id[qid] for qid in FULL_QUERY_IDS[case_id]]
        else:
            raise ValueError('mode harus pilot atau full')
        for start, end in windows:
            for batch in batches:
                group.append(task_info(case_id, batch, start, end))
        case_tasks[case_id] = group

    tasks = []
    longest = max((len(group) for group in case_tasks.values()), default=0)
    for index in range(longest):
        for case_id in DISASTER_CASES:
            if index < len(case_tasks[case_id]):
                tasks.append(case_tasks[case_id][index])
    return tasks


def preview_plan(mode=RUN_MODE):
    tasks = build_tasks(mode)
    print('=' * 68)
    print('PREVIEW-ONLY | mode:', mode, '| tugas awal:', len(tasks))
    print('Full window:', FULL_WINDOW_HOURS, 'jam | batas panggilan per sesi:', MAX_NEW_RUNS)
    print('EXECUTE_SCRAPING:', EXECUTE_SCRAPING,
        '| PAID_RUN_CONFIRMATION:', 'UNLOCKED' if PAID_RUN_CONFIRMATION else 'LOCKED')
    print('Saldo terpakai (informasi): $', CREDIT_USED_TO_DATE_USD,
        '| saldo TERSEDIA:', AVAILABLE_CREDIT_USD)
    print('Batas sesi: $', MAX_SESSION_SPEND_USD,
        '| per run: $', MAX_CHARGE_PER_RUN_USD)
    for case in DISASTER_CASES:
        print(' ', case, ':', sum(t['case'] == case for t in tasks), 'tasks')
    for task in tasks[:min(6, len(tasks))]:
        print(' -', task['case'], task['query_id'],
            iso_wib(task['start_wib']), '→', iso_wib(task['end_wib']))
    return tasks

TASK_PREVIEW = preview_plan()


PREVIEW-ONLY | mode: full | tugas awal: 416
Full window: 6 jam | batas panggilan per sesi: 3
EXECUTE_SCRAPING: False | PAID_RUN_CONFIRMATION: LOCKED
Saldo terpakai (informasi): $ 1.28 | saldo TERSEDIA: 3.72
Batas sesi: $ 0.12 | per run: $ 0.03
  lombok_2018 : 136 tasks
  palu_2018 : 136 tasks
  sumatra_2025 : 144 tasks
 - lombok_2018 E01_P01 2018-08-14 06:00:00+0700 → 2018-08-14 12:00:00+0700
 - palu_2018 E01_P01 2018-10-03 06:00:00+0700 → 2018-10-03 12:00:00+0700
 - sumatra_2025 E01_P01 2025-12-04 06:00:00+0700 → 2025-12-04 12:00:00+0700
 - lombok_2018 E01_P01 2018-08-14 12:00:00+0700 → 2018-08-14 18:00:00+0700
 - palu_2018 E01_P01 2018-10-03 12:00:00+0700 → 2018-10-03 18:00:00+0700
 - sumatra_2025 E01_P01 2025-12-04 12:00:00+0700 → 2025-12-04 18:00:00+0700


In [21]:
# 04A — PREVIEW QUERY ALTERNATIF TANPA MEMANGGIL APIFY
# Belum mengaktifkan query tambahan. Setelah hasil E01_P01 dievaluasi,
# pertimbangkan E02_P01 / E03_P01 untuk variasi lokasi.
ALTERNATIVE_QUERY_IDS = {
    'lombok_2018': ('E02_P01',),
    'palu_2018': ('E02_P01',),
    'sumatra_2025': ('E02_P01', 'E03_P01', 'E04_P01'),
}

def preview_alternative_queries():
    for case, ids in ALTERNATIVE_QUERY_IDS.items():
        indexed = {b['query_id']: b for b in DISASTER_CASES[case]['query_batches']}
        print('\n', case)
        for qid in ids:
            b = indexed[qid]
            print(' ', qid, '| EVENT:', ', '.join(b['event_terms']))
            print(' ', ' | POLICY:', ', '.join(b['policy_terms']))
    print('Tidak ada query alternatif yang dijalankan; FULL_QUERY_IDS tetap E01_P01.')

preview_alternative_queries()



 lombok_2018
  E02_P01 | EVENT: NTB, Nusa Tenggara Barat, Mataram, Lombok Utara, Lombok Barat
   | POLICY: bencana nasional, status bencana nasional, status nasional, penetapan bencana nasional, penetapan status bencana, penetapan status

 palu_2018
  E02_P01 | EVENT: Donggala, gempa Donggala, tsunami Donggala, Sulawesi Tengah, gempa Sulawesi Tengah
   | POLICY: bencana nasional, status bencana nasional, status nasional, penetapan bencana nasional, penetapan status bencana, penetapan status

 sumatra_2025
  E02_P01 | EVENT: Sumatera Utara, banjir Sumatra Utara, banjir Sumatera Utara, longsor Sumatra Utara, longsor Sumatera Utara
   | POLICY: bencana nasional, status bencana nasional, status nasional, penetapan bencana nasional, penetapan status bencana, penetapan status
  E03_P01 | EVENT: bencana Sumatra Utara, bencana Sumatera Utara, Sumatra Barat, Sumatera Barat, banjir Sumatra Barat
   | POLICY: bencana nasional, status bencana nasional, status nasional, penetapan bencana nasional,

In [22]:
# 05 — VALIDASI PRE-FLIGHT TANPA API

def preflight(tasks: list[dict], mode=RUN_MODE):
    assert len(DISASTER_CASES) == 3
    assert {c: len(v['query_batches']) for c, v in DISASTER_CASES.items()} == {
        'lombok_2018': 9, 'palu_2018': 9, 'sumatra_2025': 24,
    }
    assert MAX_NEW_RUNS > 0
    assert len({t['task_id'] for t in tasks}) == len(tasks), 'Task ID duplikat'
    for task in tasks:
        assert task['start_wib'] < task['end_wib']
        assert 'lang:in' in task['query']
        assert 'since_time:' in task['query'] and 'until_time:' in task['query']
        assert ' OR ' in task['base_query']
        assert ') (' in task['base_query'], 'Query harus EVENT dan POLICY'
        assert ' AND ' not in task['base_query'], 'Hapus literal AND dari query'
        assert task['case'] in DISASTER_CASES
        if mode == 'full':
            assert task['case'] in FULL_CASES
            assert task['query_id'] in FULL_QUERY_IDS[task['case']]
            assert (task['end_wib'] - task['start_wib']) == timedelta(hours=FULL_WINDOW_HOURS)
    return True

assert preflight(TASK_PREVIEW)
print('PREFLIGHT OFFLINE: PASS')


PREFLIGHT OFFLINE: PASS


## Langkah 3 — Validasi Actor, format tweet, dan pembagian jendela
Yang divalidasi: tweet asli (`type=tweet`), ID, `createdAt` parseable, tanggal dalam jendela,
serta dua kelompok keyword. Hasil yang meragukan tetap disimpan ke arsip raw dan audit.
Catatan: pencocokan teks adalah indikator otomatis, bukan keputusan relevansi semantik final.

In [65]:
# 06 — FUNGSI AUDIT DATA (TANPA API CALL)

def match_any_phrase(text: str, terms: list[str]) -> bool:
    # Hindari pencocokan sebagian kata (misalnya 'Palu' di 'kepaluan').
    sample = str(text).casefold()
    for term in terms:
        phrase = term.casefold()
        pat = r'(?<!\w)' + re.escape(phrase) + r'(?!\w)'
        if re.search(pat, sample):
            return True
    return False

# ALIAS LOKASI KHUSUS AUDIT
# Tidak mengubah query scraping atau task_key
AUDIT_EVENT_ALIASES = {
    'palu_2018': [
        'Sulteng',
    ],
}


def get_audit_event_terms(case_id: str) -> list[str]:
    original_terms = DISASTER_CASES[case_id]['event_terms']
    aliases = AUDIT_EVENT_ALIASES.get(case_id, [])

    return original_terms + aliases

def parse_twitter_timestamp(values: pd.Series) -> pd.Series:
    result = pd.to_datetime(
        values, format='%a %b %d %H:%M:%S %z %Y',
        errors='coerce', utc=True
    )
    # Cadangan apabila format sumber suatu hari berubah.
    missing = result.isna() & values.notna()
    if missing.any():
        result.loc[missing] = pd.to_datetime(
            values.loc[missing], errors='coerce', utc=True
        )
    return result


def audit_items(items: list[dict], task: dict) -> tuple[pd.DataFrame, dict]:
    df = pd.DataFrame(items)
    initial_columns = list(df.columns)
    for column in ['type', 'id', 'createdAt', 'text', 'lang']:
        if column not in df.columns:
            df[column] = pd.Series([None] * len(df), dtype='object')

    timestamps = parse_twitter_timestamp(df['createdAt'])
    start_utc = pd.Timestamp(task['start_wib'].astimezone(UTC))
    end_utc = pd.Timestamp(task['end_wib'].astimezone(UTC))

    df['audit_created_at_utc'] = timestamps
    df['audit_is_real_tweet'] = df['type'].eq('tweet')
    df['audit_in_window'] = timestamps.ge(start_utc) & timestamps.lt(end_utc)

    # Audit menggunakan seluruh daftar istilah yang relevan dengan kasus.
    # Term yang ada dalam batch query sendiri juga wajib dicek melalui hasil.
    event_terms = get_audit_event_terms(task['case'])
    policy_terms = DISASTER_CASES[task['case']]['policy_terms']
    texts = df['text'].fillna('').astype(str)
    df['audit_has_event'] = texts.map(lambda x: match_any_phrase(x, event_terms))
    df['audit_has_policy'] = texts.map(lambda x: match_any_phrase(x, policy_terms))
    df['audit_policy_strong'] = texts.map(lambda x: match_any_phrase(x, STRONG_POLICY_TERMS))
    df['audit_candidate'] = (
        df['audit_is_real_tweet'] & df['audit_in_window']
        & df['audit_has_event'] & df['audit_has_policy']
    )

    valid_ids = df.loc[df['audit_is_real_tweet'], 'id'].dropna().astype(str)
    n_mock = int(df['type'].eq('mock_tweet').sum())
    n_real = int(df['audit_is_real_tweet'].sum())
    n_in = int((df['audit_is_real_tweet'] & df['audit_in_window']).sum())
    n_candidate = int(df['audit_candidate'].sum())

    metrics = {
        'rows_raw': len(df),
        'rows_real_tweet': n_real,
        'rows_mock': n_mock,
        'rows_unknown_type': int((~df['type'].isin(['tweet', 'mock_tweet'])).sum()),
        'rows_real_in_window': n_in,
        'rows_real_out_window': int((df['audit_is_real_tweet'] & ~df['audit_in_window']).sum()),
        'rows_real_missing_timestamp': int((df['audit_is_real_tweet'] & timestamps.isna()).sum()),
        'rows_event_match': int((df['audit_is_real_tweet'] & df['audit_has_event']).sum()),
        'rows_policy_match': int((df['audit_is_real_tweet'] & df['audit_has_policy']).sum()),
        'rows_event_and_policy_in_window': n_candidate,
        'rows_strong_policy_candidate': int((df['audit_candidate'] & df['audit_policy_strong']).sum()),
        'unique_real_ids': int(valid_ids.nunique()),
        'unique_real_in_window_ids': int(df.loc[
            df['audit_is_real_tweet'] & df['audit_in_window'], 'id'
        ].dropna().astype(str).nunique()),
        'unique_policy_candidate_ids': int(df.loc[
            df['audit_candidate'], 'id'
        ].dropna().astype(str).nunique()),
        'duplicate_real_ids_in_dataset': int(valid_ids.duplicated().sum()),
        'raw_columns': initial_columns,
        'created_min_utc': timestamps.min().isoformat() if timestamps.notna().any() else None,
        'created_max_utc': timestamps.max().isoformat() if timestamps.notna().any() else None,
    }
    return df, metrics


def quality_flags(metrics: dict) -> list[str]:
    flags = []
    if metrics['rows_raw'] == 0:
        flags.append('EMPTY_SEARCH')
    if metrics['rows_mock']:
        flags.append('MOCK_PRESENT')
    if metrics['rows_raw'] > 0 and metrics['rows_real_tweet'] == 0:
        flags.append('NO_REAL_TWEET')
    if metrics['rows_real_out_window']:
        flags.append('OUT_OF_WINDOW')
    if metrics['rows_real_missing_timestamp']:
        flags.append('BAD_TIMESTAMP')
    if metrics['rows_real_tweet'] and metrics['rows_event_and_policy_in_window'] == 0:
        flags.append('NO_POLICY_CANDIDATE')
    # MAX_ITEMS bukan hard-limit aktual Actor. Ini indikator risiko saja.
    if metrics['unique_real_ids'] >= MAX_ITEMS:
        flags.append('POSSIBLE_TRUNCATION')
    return flags or ['BASIC_CHECKS_PASS']


# Tes cepat audit tanpa Apify dengan tanggal dan kasus otomatis.
_t = TASK_PREVIEW[0]
_event_term = DISASTER_CASES[_t['case']]['event_terms'][0]
_timedemo = (_t['start_wib'] + timedelta(minutes=2)).astimezone(UTC)
_sample = [
    {'type': 'mock_tweet', 'id': '-1', 'text': 'Actor mock placeholder'},
    {'type': 'tweet', 'id': 'demo-real',
    'text': f'Apakah {_event_term} perlu ditetapkan sebagai bencana nasional?',
    'createdAt': _timedemo.strftime('%a %b %d %H:%M:%S %z %Y'),
    'lang': 'id'},
]
_audit_demo, _m_demo = audit_items(_sample, _t)
assert _m_demo['rows_mock'] == 1 and _m_demo['rows_event_and_policy_in_window'] == 1
print('UNIT TEST AUDIT: PASS (tidak memanggil Apify)')

# UNIT TEST ALIAS LOKASI PALU
palu_terms = get_audit_event_terms('palu_2018')

assert match_any_phrase(
    'Gempa bumi dan tsunami Sulteng perlu menjadi bencana nasional',
    palu_terms
)

assert match_any_phrase(
    'Pemerintah menangani bencana di Sulawesi Tengah',
    palu_terms
)

assert not match_any_phrase(
    'Status bencana nasional perlu dipertimbangkan',
    palu_terms
)

print('UNIT TEST ALIAS PALU: PASS')


UNIT TEST AUDIT: PASS (tidak memanggil Apify)
UNIT TEST ALIAS PALU: PASS


In [66]:

# 06B — AUDIT RELEVANSI PERISTIWA (OFFLINE)
# Tidak mengubah DISASTER_CASES, query, task_id, atau checkpoint.

EVENT_SCOPE_AUDIT_VERSION = "event_scope_v1"

HISTORICAL_MARKERS = [
    "tsunami Aceh",
    "tsunami 2004",
    "Aceh 2004",
    "Sumatra-Andaman",
    "Sumatera-Andaman",
]

SUMATRA_SCOPE_LOCATIONS = [
    "Aceh", "Sumatra", "Sumatera",
    "Sumut", "Sumbar",
    "Aceh Utara", "Aceh Tamiang",
    "Tapanuli", "Agam",
]

SUMATRA_2025_DISASTER_TERMS = [
    "banjir", "banjir bandang", "longsor",
    "bencana ekologis", "bencana ekologi",
]

CURRENT_DECISION_CUES = [
    "belum", "tak kunjung", "sampai sekarang",
    "sekarang", "tak juga", "tidak juga",
    "tidak mau", "mendesak", "didesak",
    "desak", "minta", "meminta",
    "butuh", "perlu", "segera",
    "mengapa", "kenapa",
]

EXPLICIT_PRESENT_CUES = [
    "sampai sekarang", "sekarang", "saat ini",
    "hingga kini", "tak kunjung",
    "belum ditetapkan", "tahun ini",
]


def classify_event_scope(text: str, case_id: str):
    """
    Return (status_relevansi, alasan).
    Aturan heuristik konservatif, bukan label final.
    """
    text = str(text or "")

    historical = (
        bool(re.search(r"(?<!\w)2004(?!\w)", text))
        or match_any_phrase(text, HISTORICAL_MARKERS)
    )

    policy_match = match_any_phrase(
        text,
        DISASTER_CASES[case_id]["policy_terms"]
    )

    # Kasus 2018: gunakan lokasi peristiwa yang sudah ada.
    if case_id in ("lombok_2018", "palu_2018"):
        event_match = match_any_phrase(
            text, get_audit_event_terms(case_id)
        )

        if event_match and policy_match:
            if historical:
                return "TARGET_WITH_HISTORY", "case_and_history"
            return "TARGET_CURRENT", "case_and_policy"

        return "NEEDS_REVIEW", "insufficient_case_context"

    # Kasus banjir/longsor Sumatra 2025
    location = match_any_phrase(
        text, SUMATRA_SCOPE_LOCATIONS
    )

    hazard = match_any_phrase(
        text, SUMATRA_2025_DISASTER_TERMS
    )

    year_2025 = bool(
        re.search(r"(?<!\w)2025(?!\w)", text)
    )

    decision_now = (
        policy_match
        and match_any_phrase(text, CURRENT_DECISION_CUES)
    )

    strong_current = (
        location and policy_match
        and (hazard or year_2025)
    )

    weak_current = location and decision_now

    if historical:
        if strong_current:
            return (
                "TARGET_WITH_HISTORY",
                "current_disaster_and_history"
            )

        if (
            weak_current
            and match_any_phrase(text, EXPLICIT_PRESENT_CUES)
        ):
            return (
                "NEEDS_REVIEW",
                "history_with_ambiguous_current_reference"
            )

        return (
            "HISTORICAL_ONLY",
            "historical_without_2025_evidence"
        )

    if strong_current or weak_current:
        return (
            "TARGET_CURRENT",
            "current_event_or_decision_evidence"
        )

    return (
        "NEEDS_REVIEW",
        "insufficient_2025_event_evidence"
    )



# REGRESSION TEST — TANPA APIFY

test_cases = [
    (
        "Tsunami Aceh 2004 ditetapkan sebagai bencana nasional.",
        "HISTORICAL_ONLY",
    ),
    (
        "Banjir Aceh 2025 harus ditetapkan bencana nasional.",
        "TARGET_CURRENT",
    ),
    (
        "Tsunami Aceh 2004 ditetapkan sebagai bencana nasional, "
        "mengapa banjir Aceh 2025 belum?",
        "TARGET_WITH_HISTORY",
    ),
    (
        "Aceh 2004 dulu bencana nasional, "
        "sekarang mengapa status itu masih diperdebatkan?",
        "NEEDS_REVIEW",
    ),
]

for sample_text, expected in test_cases:
    result, reason = classify_event_scope(
        sample_text, "sumatra_2025"
    )

    assert result == expected, (
        f"Expected {expected}, got {result}: {sample_text}"
    )

    print(f"PASS: {result} | {reason}")

print("\nEVENT SCOPE UNIT TEST: PASS")


PASS: HISTORICAL_ONLY | historical_without_2025_evidence
PASS: TARGET_CURRENT | current_event_or_decision_evidence
PASS: TARGET_WITH_HISTORY | current_disaster_and_history
PASS: NEEDS_REVIEW | history_with_ambiguous_current_reference

EVENT SCOPE UNIT TEST: PASS


In [67]:
# 06C — EVENT SCOPE V2 (OFFLINE; V1 TIDAK DIUBAH)
# Letakkan setelah Cell 06B dan sebelum Cell 07.
# Membutuhkan: import re, html.unescape, match_any_phrase, DISASTER_CASES,
# get_audit_event_terms(), classify_event_scope() dari Cell 06B.

from html import unescape

EVENT_SCOPE_V2_VERSION = "event_scope_v2"


def _v2_normalize(value):
    """Normalisasi hanya untuk pemeriksaan; teks asli tetap disimpan."""
    return re.sub(r"\s+", " ", unescape(str(value or "")).casefold()).strip()


def _v2_re(text, pattern):
    return bool(re.search(pattern, text, flags=re.IGNORECASE))


def classify_event_scope_v2(text: str, case_id: str):
    """Klasifikasi heuristik konservatif; BUKAN gold-standard atau label sentimen.

    Return: (event_scope_v2, scope_reason_v2).
    """
    if case_id not in DISASTER_CASES:
        raise ValueError(f"Kasus tidak dikenali: {case_id}")

    sample = _v2_normalize(text)
    if not sample:
        return "NEEDS_REVIEW", "empty_text"

    # Periode lain tetap mengikuti V1; V2 kali ini difokuskan untuk
    # persoalan sejarah Aceh 2004 versus banjir Sumatra 2025.
    if case_id != "sumatra_2025":
        return classify_event_scope(sample, case_id)

    policy = match_any_phrase(sample, DISASTER_CASES[case_id]["policy_terms"])
    if not policy:
        return "NEEDS_REVIEW", "no_explicit_policy_term"

    locations = (
        "aceh", "sumatra", "sumatera", "sumut", "sumbar",
        "sumatera utara", "sumatra utara", "sumatera barat", "sumatra barat",
        "tapanuli", "tapteng", "taput", "tapsel", "medan", "agam",
        "aceh tamiang", "aceh utara", "langsa",
    )
    location = match_any_phrase(sample, list(locations))

    history = bool(
        _v2_re(sample, r"(?<!\w)(?:2004|1992|2006|2009|2020)(?!\w)")
        or _v2_re(sample, r"\btsunami\s+(?:aceh|sumatr[ae])\b")
        or _v2_re(sample, r"\bsumatr[ae][-\s]*andaman\b")
        or _v2_re(sample, r"\bcovid(?:[-\s]*19)?\b")
    )

    flood = _v2_re(sample, r"\b(?:banjir(?:\s+bandang)?|longsor|bencana\s+ekologis?)\b")
    explicit_year = _v2_re(sample, r"(?<!\w)2025(?!\w)")

    # Konteks tiga provinsi di Sumatra (termasuk ejaan nonbaku "propinsi").
    aceh = _v2_re(sample, r"\baceh\b")
    north = _v2_re(sample, r"\b(?:sumut|sumatr[ae]\s+utara|tapanuli|medan)\b")
    west = _v2_re(sample, r"\b(?:sumbar|sumatr[ae]\s+barat|padang|agam)\b")
    multi_region = (sum((aceh, north, west)) >= 3) or _v2_re(
        sample, r"\b(?:3|tiga)\s*(?:provinsi|propinsi)\b"
    )

    decision_action = _v2_re(
        sample,
        r"\b(?:belum|belom|blm|tidak|tdk|gak|ga|nggak|enggak|"
        r"bukan|tak\s+kunjung|mendesak|didesak|desak|minta|meminta|"
        r"dinaikkan|menaikkan|naikkan|naikin|tingkatkan|meningkatkan|"
        r"ditetapkan|menetapkan|tetapkan|penetapan|"
        r"kenapa|mengapa|perlukah|perlu|butuh|segera)\b"
    )
    now_cue = _v2_re(
        sample,
        r"\b(?:sekarang|skrg|skg|saat\s+ini|kini|hari\s+ini|"
        r"sampai\s+sekarang|hingga\s+kini|tahun\s+ini|"
        r"pemerintah\s+pusat\s+saat\s+ini|prabowo)\b"
    )
    contrasting = _v2_re(
        sample,
        r"\b(?:dulu|dlu|waktu\s+itu|zaman|jaman|"
        r"padahal|sedangkan|sementara|dibanding|beda|lah\s+ini|"
        r"kok|ko|sekarang|skrg)\b"
    )

    # Bukti kuat bahwa pembahasan menyasar bencana 2025.
    event_2025_evidence = location and (flood or explicit_year or multi_region)
    present_government = _v2_re(
        sample, r"\b(?:prabowo|pemerintah\s+pusat\s+saat\s+ini)\b"
    )
    policy_2025_evidence = location and decision_action and (multi_region or present_government)

    if history:
        if event_2025_evidence or policy_2025_evidence:
            return "TARGET_WITH_HISTORY", "2025_context_plus_history"
        if now_cue or contrasting or (location and decision_action and _v2_re(sample, r"\b(?:ini|tiga|3)\b")):
            return "NEEDS_REVIEW", "historical_comparison_ambiguous"
        return "HISTORICAL_ONLY", "historical_no_current_disaster_evidence"

    # Tanpa referensi sejarah, opini kebijakan di lokasi dan konteks
    # keputusan masa kini dapat tetap relevan meski tidak menyebut "banjir".
    if event_2025_evidence:
        return "TARGET_CURRENT", "2025_event_evidence"
    if location and decision_action:
        return "TARGET_CURRENT", "location_and_policy_decision"
    return "NEEDS_REVIEW", "insufficient_event_context"


# Unit test dari tweet nyata yang sebelumnya diperiksa.
# Ekspektasi ini adalah adjudikasi sementara untuk regresi, bukan skor akurasi.
V2_REGRESSION_SAMPLES = [
    (
        "1996005502111846540",
        "Ya Allah udh 7 hari kelaparan. Makanya rakyat minta ditetapkan bencana nasional "
        "Tsunami aceh aja dlu cm satu propinsi bisa jd bencana nasional, skrg 3 propinsi",
        "TARGET_WITH_HISTORY",
    ),
    (
        "1996594403432112336",
        "Kalo zaman SBY, hari ke 3 setelah bencana tsunami Aceh langsung tetapkan "
        "status bencana nasional. Pemerintah pusat saat ini hanya sekumpulan "
        "pengangguran yg mencari uang melalui kekuasaan @prabowo",
        "TARGET_WITH_HISTORY",
    ),
    (
        "1996207496550318418",
        "Sekarang dpr sudah mendesak status bencana nasional. Waktu tsunami aceh "
        "bisa ko, lah ini? 3 provinsi masa dibiarin",
        "TARGET_WITH_HISTORY",
    ),
    (
        "1994790454051713494",
        "Polemik Status Bencana Nasional: Perlukah Sumatera dan Aceh Dinaikkan?",
        "TARGET_CURRENT",
    ),
    (
        "1996227070318285049",
        "Saat ini berharap dinaikkan status bencana kami ini ke bencana nasional "
        "apalagi Aceh, Tapanuli, dan Sumbar",
        "TARGET_CURRENT",
    ),
    (
        "1996027810281128040",
        "Keuangan Pemda ACEH-SUMUT dan SUMBAR tidak akan mampu mengatasinya. "
        "Presiden umumkan bencana ini sebagai BENCANA NASIONAL",
        "TARGET_CURRENT",
    ),
    (
        "1996419143890018743",
        "Menurut pemerintahan Prabowo bencana Aceh Sumut Sumbar BUKAN Bencana Nasional",
        "TARGET_CURRENT",
    ),
    (
        "1996074461297344839",
        "Gempa tsunami Aceh 2004, banjir bandang longsor Aceh Sumut Sumbar 2025 "
        "mana yang diakui bencana nasional?",
        "TARGET_WITH_HISTORY",
    ),
    (
        "1996352009776304267",
        "Gempa tsunami Sumatra-Andaman 2004 dinyatakan sbg bencana nasional "
        "setelah 12 jam, jalur komunikasi Aceh terputus",
        "HISTORICAL_ONLY",
    ),
    (
        "1996265317547520148",
        "Tsunami Aceh sudah ditetapkan bencana nasional, momentum penyelesaian "
        "konflik dan perjanjian Helsinki",
        "HISTORICAL_ONLY",
    ),
    (
        "manual_ambigu",
        "Aceh 2004 dulu bencana nasional, sekarang mengapa status itu masih diperdebatkan?",
        "NEEDS_REVIEW",
    ),
    (
        "manual_non_case",
        "Status bencana nasional mungkin mengundang bantuan asing",
        "NEEDS_REVIEW",
    ),
]

for tweet_id, sample_text, expected in V2_REGRESSION_SAMPLES:
    result, reason = classify_event_scope_v2(sample_text, "sumatra_2025")
    assert result == expected, (
        f"Tweet {tweet_id}: expected {expected}; got {result} ({reason})"
    )

print(f"EVENT SCOPE V2 TEST: PASS ({len(V2_REGRESSION_SAMPLES)} kasus; tanpa Apify)")


EVENT SCOPE V2 TEST: PASS (12 kasus; tanpa Apify)


## Langkah 4 — Penyimpanan raw dan checkpoint
Satu folder per kasus. JSONL mempertahankan objek tweet original, audit CSV adalah hasil turunan.
Checkpoint memuat hash task (mempertimbangkan query dan waktu), run ID, dataset ID dan indikator kualitas.
Task yang sudah tersimpan tidak dipanggil ulang saat notebook dijalankan ulang.

In [68]:
# 07 — HELPER CHECKPOINT DAN PENYIMPANAN

def json_write_atomic(path: Path, items: list[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        for item in items:
            f.write(json.dumps(item, ensure_ascii=False, default=str) + '\n')
    tmp.replace(path)


def csv_write_atomic(path: Path, df: pd.DataFrame):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.tmp')
    df.to_csv(tmp, index=False, encoding='utf-8-sig')
    tmp.replace(path)


def append_checkpoint(record: dict):
    CHECKPOINT_FILE.parent.mkdir(parents=True, exist_ok=True)
    with CHECKPOINT_FILE.open('a', encoding='utf-8') as f:
        f.write(json.dumps(record, ensure_ascii=False, default=str) + '\n')
        f.flush()
        os.fsync(f.fileno())


def load_checkpoint() -> dict[str, dict]:
    newest = {}
    if not CHECKPOINT_FILE.exists():
        return newest
    with CHECKPOINT_FILE.open(encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            try:
                entry = json.loads(line)
            except json.JSONDecodeError as exc:
                raise RuntimeError('Checkpoint rusak. Periksa file sebelum lanjut.') from exc
            newest[entry['task_id']] = entry
    return newest


def make_run_record(task: dict, run_dict: dict, metrics: dict,
                    flags: list[str], raw_path: Path, audit_path: Path) -> dict:
    # SAVED = berhasil disimpan; data_quality_status = ada atau tidak kandidat.
    data_quality_status = (
        'NO_REAL_DATA' if metrics['unique_real_ids'] == 0 else
        'NO_POLICY_CANDIDATES' if metrics['unique_policy_candidate_ids'] == 0 else
        'HAS_POLICY_CANDIDATES'
    )
    return {
        'task_id': task['task_id'], 'config_version': CONFIG_VERSION,
        'status': 'SAVED', 'data_quality_status': data_quality_status,
        'case': task['case'], 'query_id': task['query_id'],
        'depth': task['depth'], 'actor_id': ACTOR_ID,
        'run_id': run_dict.get('id'),
        'dataset_id': run_dict.get('defaultDatasetId') or run_dict.get('default_dataset_id'),
        'actor_status': run_dict.get('status'),
        'query': task['query'], 'base_query': task['base_query'],
        'start_wib': iso_wib(task['start_wib']),
        'end_wib': iso_wib(task['end_wib']),
        'start_utc': task['start_wib'].astimezone(UTC).isoformat(),
        'end_utc': task['end_wib'].astimezone(UTC).isoformat(),
        'quality_flags': flags, **metrics,
        'raw_path': str(raw_path.relative_to(PROJECT_ROOT)),
        'audit_path': str(audit_path.relative_to(PROJECT_ROOT)),
        'saved_at_utc': datetime.now(UTC).isoformat(),
        'usage_total_usd': run_dict.get('usageTotalUsd', run_dict.get('usage_total_usd')),
        'max_total_charge_usd_requested': str(task.get('per_run_cap_usd', '')),
        'server_max_items_requested': MAX_ITEMS,
    }


def run_to_dict(run: Any) -> dict:
    if isinstance(run, dict):
        return run
    if hasattr(run, 'model_dump'):
        return run.model_dump(by_alias=True)
    if hasattr(run, 'to_dict'):
        return run.to_dict()
    if hasattr(run, 'default_dataset_id'):
        return {
            'default_dataset_id': run.default_dataset_id,
            'id': getattr(run, 'id', None),
            'status': getattr(run, 'status', None),
        }
    raise TypeError(f'Tipe Apify Run tidak dikenal: {type(run)}')


def get_apify_client() -> ApifyClient:
    token = os.getenv('APIFY_API_TOKEN', '').strip()
    if not token:
        token = getpass('Apify API token (tidak tampil): ').strip()
    if not token:
        raise RuntimeError('API token kosong. Tidak dapat menjalankan Actor.')
    # Jangan cetak atau simpan token dalam notebook, checkpoint, atau GitHub.
    return ApifyClient(token)


In [25]:
# 07A — PEMERIKSAAN BIAYA HISTORIS (READ-ONLY, OPSIONAL; DEFAULT OFF)
# Mengaktifkan ini hanya membaca metadata run lama. Tidak memulai scraping baru.
CHECK_HISTORICAL_COSTS = False

if CHECK_HISTORICAL_COSTS:
    _history_client = get_apify_client()
    _cost_rows = []
    for rec in load_checkpoint().values():
        if rec.get('status') != 'SAVED' or not rec.get('run_id'):
            continue
        info = run_to_dict(_history_client.run(rec['run_id']).get() or {})
        price = info.get('usageTotalUsd', info.get('usage_total_usd'))
        _cost_rows.append({
            'case': rec.get('case'), 'run_id': rec.get('run_id'),
            'rows_raw': rec.get('rows_raw'),
            'unique_ids_in_run': rec.get('unique_real_ids'),
            'usage_total_usd': price,
        })
    df_cost = pd.DataFrame(_cost_rows)
    display(df_cost)
    if not df_cost.empty:
        _charges = pd.to_numeric(df_cost['usage_total_usd'], errors='coerce')
        print('Biaya run dengan data tersedia: $', round(_charges.sum(), 5))
        print('Run tanpa data biaya:', int(_charges.isna().sum()))
        print('Jangan menganggap NaN sebagai biaya $0.')
else:
    print('Pemeriksaan biaya historis dilewati (tidak ada API call).')


Pemeriksaan biaya historis dilewati (tidak ada API call).


In [26]:
# 07B — REKONSILIASI BIAYA SMOKE TEST (OFFLINE, TANPA TOKEN API)
# Periksa: Apify Billing > Usage menampilkan $1.21 (sebelum), $1.22 (sesudah).
# Selisih dashboard adalah pemakaian PLATFORM, bukan bukti tagihan per Actor run.
USAGE_BEFORE_SMOKE_USD = Decimal('1.21')
USAGE_AFTER_SMOKE_USD = Decimal('1.22')
SMOKE_RUN_IDS = (
    'Q02DRjtfM0hRo9D5h',  # Lombok
    'NJ4LiyFPaZlHXA1wd',  # Palu
    'htS8vz5ELBdk9BGTH',  # Sumatra
)

_state_cost = load_checkpoint()
_smoke = [r for r in _state_cost.values()
          if r.get('run_id') in SMOKE_RUN_IDS and r.get('status') == 'SAVED']
assert len(_smoke) == 3, 'Run smoke test belum lengkap pada checkpoint lokal.'

df_smoke_cost = pd.DataFrame([{
    'case': r['case'],
    'run_id': r['run_id'],
    'raw': r.get('rows_raw', 0),
    'real': r.get('rows_real_tweet', 0),
    'mock': r.get('rows_mock', 0),
    'reported_run_usage_usd': r.get('usage_total_usd'),
} for r in _smoke])
display(df_smoke_cost)

_delta = USAGE_AFTER_SMOKE_USD - USAGE_BEFORE_SMOKE_USD
_run_reported = sum(Decimal(str(r.get('usage_total_usd') or 0)) for r in _smoke)
print('Selisih penggunaan platform dari dashboard: $', _delta)
print('Jumlah usage_total_usd pada checkpoint   : $', _run_reported)
if _delta != _run_reported:
    print('PERINGATAN: angka platform dan metadata run tidak sama. '
          'Jangan menafsirkan 0 pada checkpoint sebagai gratis. '
          'Cocokkan rincian Apify Billing/Usage sebelum melanjutkan.')
print('Tidak ada panggilan Actor baru pada cell ini.')


,case,run_id,raw,real,mock,reported_run_usage_usd
0,lombok_2018,Q02DRjtfM0hRo9D5h,15,0,15,0.0
1,palu_2018,NJ4LiyFPaZlHXA1wd,15,0,15,0.0
2,sumatra_2025,htS8vz5ELBdk9BGTH,15,0,15,0.0


Selisih penggunaan platform dari dashboard: $ 0.01
Jumlah usage_total_usd pada checkpoint   : $ 0
PERINGATAN: angka platform dan metadata run tidak sama. Jangan menafsirkan 0 pada checkpoint sebagai gratis. Cocokkan rincian Apify Billing/Usage sebelum melanjutkan.
Tidak ada panggilan Actor baru pada cell ini.


In [27]:
# 08 — ADAPTIVE SPLIT OPTIONAL; OFF DEFAULT
# Produksi tahap awal tidak memakai recursive split otomatis.
# Ini untuk batch selektif setelah audit, bukan bagian dari full-plan 416 task.

def split_task(task: dict) -> list[dict]:
    if task['depth'] >= MAX_ADAPTIVE_DEPTH:
        return []
    total_seconds = (task['end_wib'] - task['start_wib']).total_seconds()
    if total_seconds > 6 * 3600:
        step = timedelta(hours=6)
    elif total_seconds > 1 * 3600:
        step = timedelta(hours=1)
    elif total_seconds > 15 * 60:
        step = timedelta(minutes=15)
    else:
        return []
    children = []
    start = task['start_wib']
    while start < task['end_wib']:
        stop = min(start + step, task['end_wib'])
        children.append(task_info(
            task['case'], {'query_id': task['query_id'],
                           'base_query': task['base_query']},
            start, stop, depth=task['depth'] + 1))
        start = stop
    return children


## Langkah 5 — Konfirmasi saldo dan batch produksi

`RUN_MODE='full'` adalah **mode perencanaan**, bukan izin otomatis. Sebelum mengaktifkan:

1. Periksa **saldo tersedia aktual** di dashboard Apify; isi `AVAILABLE_CREDIT_USD` dengan nominal *sisa* (bukan $1.21 yang sudah dipakai). Sisakan buffer `$0.50` atau lebih.
2. Pastikan preview menunjukan **416 task awal** (136 Lombok + 136 Palu + 144 Sumatra), **bukan 5.904** atau seluruh 42 query batch. Dengan checkpoint lama, task yang sama akan dilewati.
3. Jalankan cell **tes offline** dan lihat pesan `PASS`. Ada juga **uji fake Actor**, jadi tidak memakai token.
4. Untuk batch pertama, gunakan `MAX_NEW_RUNS = 3`, `MAX_SESSION_SPEND_USD = Decimal('0.12')`, dan `MAX_CHARGE_PER_RUN_USD = Decimal('0.03')`.
5. Baru setelah validasi, isi `PAID_RUN_CONFIRMATION='I_APPROVE_PAID_FULL_SCRAPING'` serta `EXECUTE_SCRAPING=True`, jalankan Cell 04 dan Cell 10. **Cek saldo lagi sesudahnya; jangan gunakan Run All tanpa mengembalikan EXECUTE_SCRAPING=False.**

Guard mencegah panggilan apabila saldo kosong, konfirmasi belum diberikan, atau sisa setelah cadangan lebih kecil dari batas anggaran sesi. Batas per run bukan batas total seluruh sesi; batas sesi dijaga oleh reservasi maksimum biaya setiap run. Apify tetap dapat memiliki pembulatan atau charge tambahan, jadi saldo dashboard adalah sumber utama.

**Jika run gagal di tengah penyimpanan**, checkpoint menjadi `NEEDS_REVIEW`; notebook **tidak** mengulang task itu otomatis. Periksa run dan dataset di Apify sebelum memutuskan pemulihan.


In [28]:
# 09 — WORKER TAHAN CRASH + BATAS PER-RUN (API) + KOMITMEN PER SESI

def decimal_or_none(value):
    if value is None or value == '':
        return None
    try:
        return Decimal(str(value))
    except (ValueError, InvalidOperation):
        return None


def verify_paid_unlock():
    # Dipanggil SEBELUM get_apify_client(): tidak meminta token jika belum aman.
    if not EXECUTE_SCRAPING:
        raise RuntimeError('Preview-only: EXECUTE_SCRAPING masih False.')
    if PAID_RUN_CONFIRMATION != 'I_APPROVE_PAID_FULL_SCRAPING':
        raise RuntimeError('Konfirmasi berbayar belum diisi dengan frasa persis.')
    available = decimal_or_none(AVAILABLE_CREDIT_USD)
    if available is None or available <= 0:
        raise RuntimeError('Isi AVAILABLE_CREDIT_USD dari saldo TERSISA pada dashboard Apify.')
    if available - CREDIT_RESERVE_USD < MAX_SESSION_SPEND_USD:
        raise RuntimeError(
            'Saldo tidak cukup untuk budget sesi + cadangan. '
            f'Saldo {available}, buffer {CREDIT_RESERVE_USD}, sesi {MAX_SESSION_SPEND_USD}'
        )
    if ENABLE_ADAPTIVE:
        raise RuntimeError('Adaptive OFF wajib untuk batch produksi awal.')
    return True


def collect_one(client, task: dict, per_run_cap_usd: Decimal) -> dict:
    # Kedua batas diterapkan server-side pada API Apify, bukan hanya pada input Actor.
    run_input = {
        'searchTerms': [task['query']],
        'maxItems': MAX_ITEMS,
        'queryType': 'Latest',
    }
    started = time.monotonic()
    task = {**task, 'per_run_cap_usd': str(per_run_cap_usd)}
    run_result = client.actor(ACTOR_ID).call(
        run_input=run_input,
        max_items=MAX_ITEMS,
        max_total_charge_usd=per_run_cap_usd,
        restart_on_error=False,
    )
    if run_result is None:
        raise RuntimeError('ActorClient.call() tidak mengembalikan data run. Periksa Apify sebelum retry.')
    run_dict = run_to_dict(run_result)
    if run_dict.get('status') != 'SUCCEEDED':
        raise RuntimeError(f"Actor status: {run_dict.get('status')}; cek run di dashboard.")

    dataset_id = run_dict.get('defaultDatasetId') or run_dict.get('default_dataset_id')
    if not dataset_id:
        raise ValueError('defaultDatasetId tidak ada. Cek Apify sebelum retry.')
    items = list(client.dataset(dataset_id).iterate_items())
    if not all(isinstance(item, dict) for item in items):
        raise TypeError('Actor menghasilkan item bukan dictionary.')

    # Simpan raw dulu, tanpa membuang duplikat/mock agar audit reproduksibel.
    raw_file = RAW_DIR / task['case'] / f"{task['task_id']}.jsonl"
    json_write_atomic(raw_file, items)
    if items:
        csv_write_atomic(CSV_DIR / task['case'] / f"{task['task_id']}.csv", pd.DataFrame(items))

    df_audit, metrics = audit_items(items, task)
    flags = quality_flags(metrics)
    audit_file = AUDIT_DIR / task['case'] / f"{task['task_id']}_audit.csv"
    csv_write_atomic(audit_file, df_audit)
    record = make_run_record(task, run_dict, metrics, flags, raw_file, audit_file)
    record['elapsed_seconds'] = round(time.monotonic() - started, 2)
    append_checkpoint(record)
    return record


def run_collection(client, mode=RUN_MODE, max_new_runs=MAX_NEW_RUNS):
    tasks = build_tasks(mode)
    preflight(tasks, mode)
    verify_paid_unlock()
    state = load_checkpoint()
    reserved_upper_usd = Decimal('0')
    used = 0
    consecutive_no_real = 0
    saved = []
    blocked = 0
    errors = []

    for task in tasks:
        if used >= max_new_runs:
            break
        existing = state.get(task['task_id'])
        if existing and existing.get('status') == 'SAVED':
            continue
        if existing and existing.get('status') in {
            'NEEDS_REVIEW', 'RESERVED', 'FAILED', 'FAILED_NEEDS_REVIEW'
        }:
            blocked += 1
            continue

        # Konservatif: seluruh CAP per run direservasi untuk sesi,
        # meski biaya aktual bisa lebih rendah. Tidak ada budget carried-over.
        remaining = MAX_SESSION_SPEND_USD - reserved_upper_usd
        run_cap = min(MAX_CHARGE_PER_RUN_USD, remaining)
        if run_cap < MAX_CHARGE_PER_RUN_USD:
            print('STOP: batas komitmen biaya sesi tercapai.')
            break
        assert run_cap > 0
        used += 1
        reserved_upper_usd += run_cap
        print(f"\n[{used}/{max_new_runs}] {task['case']} {task['query_id']}"
              f" | {iso_wib(task['start_wib'])} → {iso_wib(task['end_wib'])}")
        print('  Cap maksimum run:', f'${run_cap}',
              '| cap kumulatif sesi:', f'${reserved_upper_usd}')

        # Selesaikan review manual bila terjadi crash, karena pemanggilan mungkin sudah ditagih.
        start_record = {
            'task_id': task['task_id'], 'status': 'RESERVED',
            'case': task['case'], 'query_id': task['query_id'],
            'query': task['query'], 'actor_id': ACTOR_ID,
            'max_total_charge_usd_requested': str(run_cap),
            'timestamp_utc': datetime.now(UTC).isoformat(),
        }
        append_checkpoint(start_record)
        state[task['task_id']] = start_record
        try:
            record = collect_one(client, task, per_run_cap_usd=run_cap)
            state[task['task_id']] = record
            saved.append(record)
            print('  Raw:', record['rows_raw'], '| ID unik dalam run:',
                  record['unique_real_ids'], '| mock:', record['rows_mock'])
            print('  Quality:', ', '.join(record['quality_flags']))
            print('  Data quality:', record['data_quality_status'],
                  '| ID kandidat unik dalam run:', record['unique_policy_candidate_ids'])
            if record['rows_real_tweet'] == 0:
                consecutive_no_real += 1
            else:
                consecutive_no_real = 0
            print('  Run baru tanpa tweet asli berturut-turut:', consecutive_no_real)
            cost = decimal_or_none(record.get('usage_total_usd'))
            if cost is not None:
                print('  Usage run tercatat (USD):', cost)
                if cost > run_cap:
                    print('  PERINGATAN: penggunaan melampaui cap yang diminta; cek dashboard!')
            else:
                print('  Usage run belum tersedia; cek dashboard Apify.')
            if consecutive_no_real >= MAX_CONSECUTIVE_NO_REAL:
                print('PAUSE: terlalu banyak run baru tanpa tweet asli. '
                      'Periksa urutan tanggal dan query sebelum batch berikutnya.')
                break
        except Exception as exc:
            entry = {
                'task_id': task['task_id'], 'status': 'NEEDS_REVIEW',
                'case': task['case'], 'query_id': task['query_id'],
                'query': task['query'],
                'max_total_charge_usd_requested': str(run_cap),
                'timestamp_utc': datetime.now(UTC).isoformat(),
                'error': f'{type(exc).__name__}: {exc}',
            }
            append_checkpoint(entry)
            state[task['task_id']] = entry
            errors.append(entry)
            print('  STOP / NEEDS_REVIEW:', entry['error'])
            print('  Jangan retry otomatis. Periksa Apify run terlebih dahulu.')
            break   # fail-closed: stop sesi setelah error ambigu
        finally:
            if REQUEST_DELAY_SECONDS > 0 and used < max_new_runs:
                time.sleep(REQUEST_DELAY_SECONDS)

    print('\nSELESAI: Actor calls baru:', used, '| disimpan:', len(saved),
          '| perlu review:', len(errors), '| task ambigu lama dilewati:', blocked)
    print('Cap komitmen sesi:', f'${reserved_upper_usd}',
          '| jangan mulai batch berikut tanpa cek sisa saldo aktual.')
    return saved, errors


In [ ]:
# 09B — TEST REGRESI TANPA AKSES INTERNET / TANPA TOKEN APIFY
# Tes ini tidak mengubah checkpoint penelitian sungguhan.
from tempfile import TemporaryDirectory

_plan = build_tasks('full')
assert preflight(_plan, 'full')
assert len(_plan) == 416, f'Task full bukan 416: {len(_plan)}'
assert {c: sum(t['case'] == c for t in _plan) for c in FULL_CASES} == {
    'lombok_2018': 136, 'palu_2018': 136, 'sumatra_2025': 144,
}
assert all(t['query_id'] == 'E01_P01' for t in _plan)
assert len({t['task_id'] for t in _plan}) == len(_plan)
assert all(t['end_wib'] - t['start_wib'] == timedelta(hours=6) for t in _plan)

# Cross-check ke checkpoint yang sudah terbukti dari pilot (P5 dan P8part2).
_sconf = DISASTER_CASES['sumatra_2025']
_batch = _sconf['query_batches'][0]
assert task_info('sumatra_2025', _batch,
                wib_dt('2025-12-03', 6), wib_dt('2025-12-03', 12))['task_id'] == (
    'sumatra_2025_4f7c9d0f6659448b9b30'
)
assert task_info('sumatra_2025', _batch,
                wib_dt('2025-12-03', 6), wib_dt('2025-12-03', 9))['task_id'] == (
    'sumatra_2025_92a73c19df027e14390c'
)

# Verifikasi benar-benar skip checkpoint lama dalam rencana full (khusus exact task).
_old_state = load_checkpoint()
_old_saved_count = sum(_old_state.get(t['task_id'], {}).get('status') == 'SAVED'
                    for t in _plan)
print('Task yang cocok checkpoint pilot dan akan dilewati:', _old_saved_count)

# Simulasi Actor OFFLINE. Tidak ada panggilan dari apify_client.
class _FakeActor:
    def __init__(self, parent):
        self.parent = parent
    def call(self, **kwargs):
        self.parent.calls.append(kwargs)
        return {'id': 'fake-offline-run', 'status': 'SUCCEEDED',
                'defaultDatasetId': 'fake-offline-ds', 'usageTotalUsd': 0.001}

class _FakeDataset:
    def __init__(self, items):
        self.items = items
    def iterate_items(self):
        return iter(self.items)

class _FakeClient:
    def __init__(self, items):
        self.items = items
        self.calls = []
    def actor(self, actor_id):
        assert actor_id == ACTOR_ID
        return _FakeActor(self)
    def dataset(self, dataset_id):
        assert dataset_id == 'fake-offline-ds'
        return _FakeDataset(self.items)

_saved_globals = {k: globals()[k] for k in ('PROJECT_ROOT', 'RAW_DIR', 'CSV_DIR',
                                            'AUDIT_DIR', 'CHECKPOINT_FILE')}
try:
    with TemporaryDirectory() as _td:
        _root = Path(_td)
        PROJECT_ROOT = _root
        RAW_DIR = _root / '01_Raw Data' / 'outputs' / 'raw_jsonl'
        CSV_DIR = _root / '01_Raw Data' / 'outputs' / 'raw_csv'
        AUDIT_DIR = _root / '01_Raw Data' / 'outputs' / 'audit'
        CHECKPOINT_FILE = _root / '01_Raw Data' / 'outputs' / 'state' / 'collection_checkpoint.jsonl'
        _fake_task = _plan[0]
        _fake_dt = (_fake_task['start_wib'] + timedelta(minutes=1)).astimezone(UTC)
        _fake_event = DISASTER_CASES[_fake_task['case']]['event_terms'][0]
        _fake_tweet = {'type': 'tweet', 'id': 'offline-test-1',
                    'createdAt': _fake_dt.strftime('%a %b %d %H:%M:%S %z %Y'),
                    'text': f'{_fake_event} semestinya bencana nasional', 'lang': 'id'}
        _fake_client = _FakeClient([_fake_tweet, dict(_fake_tweet),
                                    {'type': 'mock_tweet', 'id': -1, 'text': 'mock'}])
        _fakerecord = collect_one(_fake_client, _fake_task, Decimal('0.03'))
        assert _fakerecord['rows_raw'] == 3
        assert _fakerecord['rows_mock'] == 1
        assert _fakerecord['unique_real_ids'] == 1
        assert _fakerecord['duplicate_real_ids_in_dataset'] == 1
        assert load_checkpoint()[_fake_task['task_id']]['status'] == 'SAVED'
        _call = _fake_client.calls[0]
        assert _call['max_items'] == 20
        assert _call['max_total_charge_usd'] == Decimal('0.03')
        assert _call['run_input']['maxItems'] == 20
        assert _call['restart_on_error'] is False
finally:
    globals().update(_saved_globals)

print('UNIT TEST OFFLINE: PASS — 416 planned, hash kompatibel, fake Actor + checkpoint aman')


Task yang cocok checkpoint pilot dan akan dilewati: 15
UNIT TEST OFFLINE: PASS — 416 planned, hash kompatibel, fake Actor + checkpoint aman


In [34]:
#09C — TEST REGRESI TANPA AKSES INTERNET / TANPA TOKEN APIFY
# TEST AUTO-PAUSE OFFLINE — TIDAK MEMANGGIL APIFY

_saved_settings = {
    name: globals()[name]
    for name in (
        'PROJECT_ROOT', 'RAW_DIR', 'CSV_DIR',
        'AUDIT_DIR', 'CHECKPOINT_FILE',
        'EXECUTE_SCRAPING', 'PAID_RUN_CONFIRMATION'
    )
}

try:
    with TemporaryDirectory() as tmp:
        root = Path(tmp)

        PROJECT_ROOT = root
        RAW_DIR = root / 'raw_jsonl'
        CSV_DIR = root / 'raw_csv'
        AUDIT_DIR = root / 'audit'
        CHECKPOINT_FILE = root / 'checkpoint.jsonl'

        # Membuka gerbang sementara untuk FakeClient saja.
        EXECUTE_SCRAPING = True
        PAID_RUN_CONFIRMATION = 'I_APPROVE_PAID_FULL_SCRAPING'

        fake_client = _FakeClient([
            {
                'type': 'mock_tweet',
                'id': -1,
                'text': 'Simulasi hasil kosong'
            }
        ])

        saved, errors = run_collection(
            fake_client,
            mode='full',
            max_new_runs=10
        )

        assert len(fake_client.calls) == 3
        assert len(saved) == 3
        assert not errors
        assert all(
            r['data_quality_status'] == 'NO_REAL_DATA'
            for r in saved
        )

        print('AUTO-PAUSE OFFLINE: PASS')

finally:
    globals().update(_saved_settings)


[1/10] lombok_2018 E01_P01 | 2018-08-14 06:00:00+0700 → 2018-08-14 12:00:00+0700
  Cap maksimum run: $0.03 | cap kumulatif sesi: $0.03
  Raw: 1 | ID unik dalam run: 0 | mock: 1
  Quality: MOCK_PRESENT, NO_REAL_TWEET
  Data quality: NO_REAL_DATA | ID kandidat unik dalam run: 0
  Run baru tanpa tweet asli berturut-turut: 1
  Usage run tercatat (USD): 0.001

[2/10] palu_2018 E01_P01 | 2018-10-03 06:00:00+0700 → 2018-10-03 12:00:00+0700
  Cap maksimum run: $0.03 | cap kumulatif sesi: $0.06
  Raw: 1 | ID unik dalam run: 0 | mock: 1
  Quality: MOCK_PRESENT, NO_REAL_TWEET
  Data quality: NO_REAL_DATA | ID kandidat unik dalam run: 0
  Run baru tanpa tweet asli berturut-turut: 2
  Usage run tercatat (USD): 0.001

[3/10] sumatra_2025 E01_P01 | 2025-12-04 06:00:00+0700 → 2025-12-04 12:00:00+0700
  Cap maksimum run: $0.03 | cap kumulatif sesi: $0.09
  Raw: 1 | ID unik dalam run: 0 | mock: 1
  Quality: MOCK_PRESENT, NO_REAL_TWEET
  Data quality: NO_REAL_DATA | ID kandidat unik dalam run: 0
  Run b

In [51]:
# 10 — GERBANG EKSEKUSI: DEFAULT TIDAK BERBAYAR
print('\nGerbang eksekusi')
print('RUN_MODE:', RUN_MODE, '| EXECUTE_SCRAPING:', EXECUTE_SCRAPING)
print('MAX_NEW_RUNS:', MAX_NEW_RUNS, '| ENABLE_ADAPTIVE:', ENABLE_ADAPTIVE)
if EXECUTE_SCRAPING:
    verify_paid_unlock()  # Fail-closed sebelum token diminta atau panggilan dibuat.
    _client = get_apify_client()
    _saved, _errors = run_collection(_client, mode=RUN_MODE, max_new_runs=MAX_NEW_RUNS)
else:
    print('PREVIEW ONLY — tidak ada token atau panggilan Actor berbayar.')



Gerbang eksekusi
RUN_MODE: full | EXECUTE_SCRAPING: True
MAX_NEW_RUNS: 3 | ENABLE_ADAPTIVE: False

[1/3] lombok_2018 E01_P01 | 2018-08-14 00:00:00+0700 → 2018-08-14 06:00:00+0700
  Cap maksimum run: $0.03 | cap kumulatif sesi: $0.03


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:zBlZ3m6qB5YGgwzZG] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:zBlZ3m6qB5YGgwzZG] -> 2026-10-11T10:52:16.743Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:zBlZ3m6qB5YGgwzZG] -> 2026-10-11T10:52:16.746Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:zBlZ3m6qB5YGgwzZG] -> 2026-10-11T10:52:16.806Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:zBlZ3m6qB5YGgwzZG] -> 2026-10-11T10:52:16.810Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:zBlZ3m6qB5YGgwzZG] -> 2026-10-11T10:52:23.319Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

  Raw: 15 | ID unik dalam run: 1 | mock: 14
  Quality: MOCK_PRESENT
  Data quality: HAS_POLICY_CANDIDATES | ID kandidat unik dalam run: 1
  Run baru tanpa tweet asli berturut-turut: 0
  Usage run tercatat (USD): 0.00375

[2/3] palu_2018 E01_P01 | 2018-10-03 00:00:00+0700 → 2018-10-03 06:00:00+0700
  Cap maksimum run: $0.03 | cap kumulatif sesi: $0.06


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:lexPDbOhKa4umXRze] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:lexPDbOhKa4umXRze] -> 2026-10-11T10:52:45.688Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:lexPDbOhKa4umXRze] -> 2026-10-11T10:52:45.691Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:lexPDbOhKa4umXRze] -> 2026-10-11T10:52:45.756Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:lexPDbOhKa4umXRze] -> 2026-10-11T10:52:45.757Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:lexPDbOhKa4umXRze] -> 2026-10-11T10:52:51.945Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

  Raw: 18 | ID unik dalam run: 3 | mock: 12
  Quality: MOCK_PRESENT
  Data quality: HAS_POLICY_CANDIDATES | ID kandidat unik dalam run: 3
  Run baru tanpa tweet asli berturut-turut: 0
  Usage run tercatat (USD): 0.0045000000000000005

[3/3] sumatra_2025 E01_P01 | 2025-12-04 00:00:00+0700 → 2025-12-04 06:00:00+0700
  Cap maksimum run: $0.03 | cap kumulatif sesi: $0.09


[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:MvQ6MLUZMsTYDoJtA] -> Status: RUNNING, Message: 
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:MvQ6MLUZMsTYDoJtA] -> 2026-10-11T10:53:17.776Z ACTOR: Pulling container image of build 0n0YJ6lOK1MKTzsdX from registry.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:MvQ6MLUZMsTYDoJtA] -> 2026-10-11T10:53:17.783Z ACTOR: Creating container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:MvQ6MLUZMsTYDoJtA] -> 2026-10-11T10:53:17.890Z ACTOR: Starting container.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:MvQ6MLUZMsTYDoJtA] -> 2026-10-11T10:53:17.891Z ACTOR: Running under "LIMITED_PERMISSIONS".
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest runId:MvQ6MLUZMsTYDoJtA] -> 2026-10-11T10:53:24.508Z Actor is running on the Apify platform, `disable_browser_sandbox` was changed to True.
[apify.twitter-x-data-tweet-scraper-pay-per-result-cheapest ru

  Raw: 21 | ID unik dalam run: 6 | mock: 9
  Quality: MOCK_PRESENT
  Data quality: HAS_POLICY_CANDIDATES | ID kandidat unik dalam run: 6
  Run baru tanpa tweet asli berturut-turut: 0
  Usage run tercatat (USD): 0.0015

SELESAI: Actor calls baru: 3 | disimpan: 3 | perlu review: 0 | task ambigu lama dilewati: 0
Cap komitmen sesi: $0.09 | jangan mulai batch berikut tanpa cek sisa saldo aktual.


In [31]:
# 11 — LAPORAN PROGRES GLOBAL (TANPA API): DE-DUP ID + TEMPORAL COVERAGE

def report_full_progress(mode='full'):
    state = load_checkpoint()
    planned = build_tasks(mode)
    planned_by_id = {t['task_id']: t for t in planned}
    statuses = Counter(state.get(tid, {}).get('status', 'PENDING') for tid in planned_by_id)
    print('\n=== FULL PROGRESS ===')
    print('Task direncanakan:', len(planned), '| status:', dict(statuses))
    by_case = {}
    all_ids = set()
    for case in FULL_CASES:
        case_ids = set()
        kept = 0
        mock = 0
        raw = 0
        missing = []
        for task in (t for t in planned if t['case'] == case):
            record = state.get(task['task_id'])
            if not record or record.get('status') != 'SAVED':
                continue
            rel = record.get('raw_path')
            if not rel:
                missing.append(task['task_id'])
                continue
            raw_path = PROJECT_ROOT / Path(str(rel).replace('\\', '/'))
            if not raw_path.is_file():
                missing.append(task['task_id'])
                continue
            with raw_path.open('r', encoding='utf-8') as f:
                for line in f:
                    if not line.strip():
                        continue
                    obj = json.loads(line)
                    raw += 1
                    if obj.get('type') == 'mock_tweet':
                        mock += 1
                    if obj.get('type') == 'tweet' and obj.get('id') not in (None, -1, '-1'):
                        kept += 1
                        case_ids.add(str(obj['id']))
        all_ids.update(case_ids)
        by_case[case] = {
            'case': case, 'raw': raw, 'mock': mock, 'real_rows': kept,
            'unique_ids': len(case_ids), 'missing_saved_raw_files': len(missing),
        }
    display(pd.DataFrame(list(by_case.values())))
    print('ID asli unik global dalam task FULL yang sudah tersimpan:', len(all_ids))
    print('NOTE: sebagian run pilot non-6-jam tidak termasuk rencana FULL ini.')
    print('NOTE: ID unik belum tentu relevan semantik atau berlabel sentimen.')

    coverage = []
    for case in FULL_CASES:
        for day in sorted({t['start_wib'].date() for t in planned if t['case'] == case}):
            daily = [t for t in planned if t['case'] == case and t['start_wib'].date() == day]
            done = sum(state.get(t['task_id'], {}).get('status') == 'SAVED' for t in daily)
            coverage.append({
                'case': case, 'date_wib': str(day),
                'saved_windows': done, 'planned_windows': len(daily),
                'coverage_complete': done == len(daily),
            })
    df_coverage = pd.DataFrame(coverage)
    dest = REPORT_DIR / 'full_coverage_grid.csv'
    csv_write_atomic(dest, df_coverage)
    print('Coverage grid (offline) tersimpan:', dest)
    if not df_coverage.empty:
        print('Hari selesai 4/4 window:', int(df_coverage['coverage_complete'].sum()),
              'dari', len(df_coverage), 'hari-kasus')
    return by_case, df_coverage

# Cukup panggil report_full_progress() SETELAH cell tes berhasil,
# tanpa harus menjalankan scraping atau mengisi API token.


In [48]:
# 11A — LAPORAN KUALITAS GLOBAL: SEMUA RUN SAVED (PILOT + FULL)
# Dapat dijalankan tanpa API. Tidak mengubah status SAVED atau menghapus mock.

def report_global_data_quality():
    state = load_checkpoint()
    real_by_case = {case: set() for case in FULL_CASES}
    candidates_by_case = {case: set() for case in FULL_CASES}
    records = []
    for rec in state.values():
        if rec.get('status') != 'SAVED' or rec.get('case') not in FULL_CASES:
            continue
        case = rec['case']
        audit_rel = rec.get('audit_path')
        missing_audit = False
        if audit_rel:
            audit_path = PROJECT_ROOT / Path(str(audit_rel).replace('\\', '/'))
            if audit_path.is_file():
                df = pd.read_csv(audit_path, dtype={'id': 'string'}, keep_default_na=False)
                if 'id' in df and 'type' in df:
                    ids = df['id'].astype(str).str.strip()
                    real_mask = df['type'].eq('tweet') & ids.ne('') & ids.ne('-1')
                    real_by_case[case].update(ids.loc[real_mask].tolist())
                    if 'audit_candidate' in df:
                        candidate_mask = (
                            df['audit_candidate']
                            .astype(str)
                            .str.lower()
                            .eq('true')
                        )

                        # Koreksi historis khusus Palu:
                        # tweet bertopik Sulteng yang sebelumnya tidak dikenali
                        if case == 'palu_2018':
                            alias_match = df['text'].fillna('').astype(str).map(
                                lambda text: match_any_phrase(
                                    text,
                                    AUDIT_EVENT_ALIASES['palu_2018']
                                )
                            )

                            in_window = (
                                df['audit_in_window']
                                .astype(str)
                                .str.lower()
                                .eq('true')
                            )

                            policy_match = (
                                df['audit_has_policy']
                                .astype(str)
                                .str.lower()
                                .eq('true')
                            )

                            candidate_mask = (
                                candidate_mask
                                | (real_mask & in_window & policy_match & alias_match)
                            )

                        candidates_by_case[case].update(
                            ids.loc[real_mask & candidate_mask].tolist()
                        )
                else:
                    missing_audit = True
            else:
                missing_audit = True
        else:
            missing_audit = True
        quality = (
            'NO_REAL_DATA' if rec.get('unique_real_ids', 0) == 0 else
            'NO_POLICY_CANDIDATES' if rec.get('rows_event_and_policy_in_window', 0) == 0 else
            'HAS_POLICY_CANDIDATES'
        )
        records.append({
            'case': case, 'task_id': rec.get('task_id'), 'query_id': rec.get('query_id'),
            'status': rec['status'], 'data_quality_status': quality,
            'rows_raw': rec.get('rows_raw', 0), 'rows_mock': rec.get('rows_mock', 0),
            'unique_real_ids_in_run': rec.get('unique_real_ids', 0),
            'unique_policy_candidate_ids_in_run': rec.get('unique_policy_candidate_ids'),
            'audit_missing': missing_audit,
        })
    df_run = pd.DataFrame(records)
    summary = []
    for case in FULL_CASES:
        subset = df_run.loc[df_run['case'].eq(case)] if not df_run.empty else pd.DataFrame()
        summary.append({
            'case': case,
            'saved_tasks': len(subset),
            'saved_but_zero_real': int(subset['data_quality_status'].eq('NO_REAL_DATA').sum()) if len(subset) else 0,
            'global_unique_real_ids': len(real_by_case[case]),
            'global_unique_policy_candidates_auto': len(candidates_by_case[case]),
            'audit_files_missing': int(subset['audit_missing'].sum()) if len(subset) else 0,
        })
    df_summary = pd.DataFrame(summary)
    csv_write_atomic(REPORT_DIR / 'collection_quality_by_run.csv', df_run)
    csv_write_atomic(REPORT_DIR / 'collection_quality_by_case.csv', df_summary)
    display(df_summary)
    print('ID asli unik GLOBAL (semua kasus):', len(set().union(*real_by_case.values())))
    print('Kandidat keyword GLOBAL (perlu validasi semantik):',
        len(set().union(*candidates_by_case.values())))
    if df_summary['audit_files_missing'].sum() > 0:
        print('PERHATIAN: audit tidak lengkap. Jangan gunakan total unik ini sebagai angka final.')
    print('Catatan: SAVED = arsip sukses; kandidat otomatis bukan opini yang sudah diberi label.')
    return df_run, df_summary

# Jalankan manual setelah selesai batch: report_global_data_quality()


In [53]:
# 11B — LAPORAN KUALITAS GLOBAL
df_run_quality, df_case_quality = report_global_data_quality()

display(df_case_quality)

,case,saved_tasks,saved_but_zero_real,global_unique_real_ids,global_unique_policy_candidates_auto,audit_files_missing
0,lombok_2018,12,2,53,52,0
1,palu_2018,12,1,60,57,0
2,sumatra_2025,15,2,119,115,0


ID asli unik GLOBAL (semua kasus): 232
Kandidat keyword GLOBAL (perlu validasi semantik): 224
Catatan: SAVED = arsip sukses; kandidat otomatis bukan opini yang sudah diberi label.


,case,saved_tasks,saved_but_zero_real,global_unique_real_ids,global_unique_policy_candidates_auto,audit_files_missing
0,lombok_2018,12,2,53,52,0
1,palu_2018,12,1,60,57,0
2,sumatra_2025,15,2,119,115,0


In [69]:

# 11C — LAPORAN RELEVANSI PERISTIWA GLOBAL (OFFLINE)

def report_event_scope_v1():
    state = load_checkpoint()
    records = []
    missing_files = []

    for rec in state.values():
        if rec.get("status") != "SAVED":
            continue

        case_id = rec.get("case")
        if case_id not in DISASTER_CASES:
            continue

        audit_rel = rec.get("audit_path")
        audit_path = (
            PROJECT_ROOT / Path(str(audit_rel).replace("\\", "/"))
            if audit_rel else None
        )

        if audit_path is None or not audit_path.is_file():
            missing_files.append(str(audit_path))
            continue

        df = pd.read_csv(
            audit_path,
            dtype={"id": "string"},
            keep_default_na=False,
        )

        required = {"id", "type", "text", "audit_in_window"}
        missing_cols = required - set(df.columns)

        if missing_cols:
            raise ValueError(
                f"Kolom audit hilang pada {audit_path}: {missing_cols}"
            )

        ids = df["id"].astype(str).str.strip()
        in_window = (
            df["audit_in_window"]
            .astype(str)
            .str.lower()
            .eq("true")
        )

        valid = (
            df["type"].eq("tweet")
            & ids.ne("")
            & ids.ne("-1")
            & in_window
        )

        event_terms = get_audit_event_terms(case_id)

        # Alias Sumatra hanya untuk audit turunan,
        # bukan untuk mengubah query scraping.
        if case_id == "sumatra_2025":
            event_terms = (
                event_terms + SUMATRA_SCOPE_LOCATIONS
            )

        policy_terms = DISASTER_CASES[case_id]["policy_terms"]

        for item in df.loc[valid].to_dict("records"):
            text = str(item["text"])

            status, reason = classify_event_scope(
                text, case_id
            )

            keyword_candidate = (
                match_any_phrase(text, event_terms)
                and match_any_phrase(text, policy_terms)
            )

            records.append({
                "audit_version": EVENT_SCOPE_AUDIT_VERSION,
                "case": case_id,
                "id": str(item["id"]).strip(),
                "text": text,
                "keyword_candidate": keyword_candidate,
                "event_scope": status,
                "scope_reason": reason,
                "task_id": rec["task_id"],
                "run_id": rec.get("run_id"),
            })

    # Jangan menulis laporan parsial jika file audit hilang.
    if missing_files:
        raise RuntimeError(
            f"{len(missing_files)} audit CSV tidak ditemukan. "
            f"Contoh: {missing_files[:3]}"
        )

    if not records:
        raise RuntimeError(
            "Belum ada tweet asli dalam audit CSV."
        )

    detail = pd.DataFrame(records)

    # Utamakan rekaman yang memenuhi keyword dan
    # status yang lebih informatif saat ada ID duplikat.
    rank = {
        "TARGET_WITH_HISTORY": 4,
        "TARGET_CURRENT": 3,
        "NEEDS_REVIEW": 2,
        "HISTORICAL_ONLY": 1,
    }

    detail["_rank"] = (
        detail["keyword_candidate"].astype(int) * 10
        + detail["event_scope"].map(rank).fillna(0)
    )

    detail = (
        detail.sort_values(
            "_rank", ascending=False, kind="stable"
        )
        .drop_duplicates(["case", "id"])
        .drop(columns="_rank")
        .reset_index(drop=True)
    )

    summary = []

    for case_id in DISASTER_CASES:
        case_df = detail.loc[detail["case"].eq(case_id)]
        candidates = case_df.loc[
            case_df["keyword_candidate"]
        ]

        counts = candidates["event_scope"].value_counts()

        current = int(counts.get("TARGET_CURRENT", 0))
        with_history = int(
            counts.get("TARGET_WITH_HISTORY", 0)
        )

        summary.append({
            "case": case_id,
            "unique_real_in_window": len(case_df),
            "keyword_candidates_rechecked": len(candidates),
            "target_current": current,
            "target_with_history": with_history,
            "historical_only": int(
                counts.get("HISTORICAL_ONLY", 0)
            ),
            "needs_review": int(
                counts.get("NEEDS_REVIEW", 0)
            ),
            "event_relevant_auto": current + with_history,
        })

    df_summary = pd.DataFrame(summary)

    csv_write_atomic(
        REPORT_DIR / "event_scope_v1_by_tweet.csv",
        detail
    )

    csv_write_atomic(
        REPORT_DIR / "event_scope_v1_by_case.csv",
        df_summary
    )

    display(df_summary)

    print(
        "Tweet unik (dalam window):",
        len(detail)
    )
    print(
        "Kandidat peristiwa otomatis:",
        df_summary["event_relevant_auto"].sum()
    )
    print(
        "Perlu pemeriksaan manual:",
        df_summary["needs_review"].sum()
    )

    return detail, df_summary


df_event_scope, df_event_scope_summary = report_event_scope_v1()


,case,unique_real_in_window,keyword_candidates_rechecked,target_current,target_with_history,historical_only,needs_review,event_relevant_auto
0,lombok_2018,53,52,52,0,0,0,52
1,palu_2018,60,57,56,1,0,0,57
2,sumatra_2025,119,115,69,1,20,25,70


Tweet unik (dalam window): 232
Kandidat peristiwa otomatis: 179
Perlu pemeriksaan manual: 25


In [70]:
# 11D — EVENT SCOPE V2: LAPORAN, PERBANDINGAN V1, DAN REVIEW MANUAL
# Letakkan setelah Cell 11C. Jalankan saat EXECUTE_SCRAPING == False.
# Input: event_scope_v1_by_tweet.csv (TIDAK ditimpa).
from IPython.display import display
import hashlib


def report_event_scope_v2():
    v1_file = REPORT_DIR / "event_scope_v1_by_tweet.csv"
    if not v1_file.is_file():
        raise FileNotFoundError(
            "Laporan V1 tidak ditemukan. Jalankan Cell 11C (OFFLINE) lebih dahulu."
        )

    df = pd.read_csv(
        v1_file, dtype={"id": "string"}, keep_default_na=False
    )
    needed = {
        "case", "id", "text", "keyword_candidate", "event_scope",
        "scope_reason", "task_id", "run_id",
    }
    if missing := needed - set(df.columns):
        raise ValueError(f"Kolom wajib V1 hilang: {sorted(missing)}")
    if df.duplicated(["case", "id"]).any():
        raise ValueError("File V1 memiliki duplikasi (case, id). Periksa sebelum lanjut.")

    # Simpan hasil V1 persis sebagai basis perbandingan; jangan tulis ulang CSV V1.
    df = df.rename(columns={
        "event_scope": "v1_scope", "scope_reason": "v1_reason"
    }).copy()
    df["keyword_candidate"] = (
        df["keyword_candidate"].astype(str).str.lower().eq("true")
    )
    df["v2_version"] = EVENT_SCOPE_V2_VERSION

    v2 = [
        classify_event_scope_v2(row.text, row.case)
        for row in df.itertuples(index=False)
    ]
    df["v2_scope"] = [x[0] for x in v2]
    df["v2_reason"] = [x[1] for x in v2]
    df["v1_to_v2_changed"] = df["v1_scope"].ne(df["v2_scope"])

    # Tambah waktu unggahan dari audit asli berdasarkan (case, id).
    # V1 tetap apa adanya; raw/checkpoint/audit asli tidak ditimpa.
    state = load_checkpoint()
    created_at = {}
    missing_audits = []
    for task in state.values():
        if task.get("status") != "SAVED":
            continue
        path = task.get("audit_path")
        if not path:
            missing_audits.append(str(task.get("task_id")))
            continue
        audit_file = PROJECT_ROOT / Path(str(path).replace("\\", "/"))
        if not audit_file.is_file():
            missing_audits.append(str(audit_file))
            continue
        source = pd.read_csv(
            audit_file,
            dtype={"id": "string"},
            keep_default_na=False,
            usecols=lambda c: c in ("id", "type", "createdAt"),
        )
        if not {"id", "type", "createdAt"}.issubset(source.columns):
            raise ValueError(f"Kolom id/type/createdAt hilang: {audit_file}")
        for row in source.loc[source["type"].eq("tweet")].itertuples(index=False):
            key = (str(task["case"]), str(row.id).strip())
            if key not in created_at and str(row.createdAt).strip():
                created_at[key] = str(row.createdAt)

    if missing_audits:
        raise RuntimeError(
            f"{len(missing_audits)} audit historis hilang; "
            f"contoh: {missing_audits[:2]}"
        )

    df["created_at_source"] = [
        created_at.get((str(row.case), str(row.id)), "")
        for row in df.itertuples(index=False)
    ]

    # Ringkasan hanya untuk tweet yang lolos keyword V1 (agar sebanding).
    cand = df.loc[df["keyword_candidate"]].copy()
    cases = list(DISASTER_CASES)
    summary = []
    for case in cases:
        group = cand.loc[cand["case"].eq(case)]
        all_case = df.loc[df["case"].eq(case)]
        counts = group["v2_scope"].value_counts()
        summary.append({
            "case": case,
            "unique_real": len(all_case),
            "keyword_candidates": len(group),
            "target_current": int(counts.get("TARGET_CURRENT", 0)),
            "target_with_history": int(counts.get("TARGET_WITH_HISTORY", 0)),
            "historical_only": int(counts.get("HISTORICAL_ONLY", 0)),
            "needs_review": int(counts.get("NEEDS_REVIEW", 0)),
            "event_relevant_auto": int(
                counts.get("TARGET_CURRENT", 0)
                + counts.get("TARGET_WITH_HISTORY", 0)
            ),
            "changed_vs_v1": int(group["v1_to_v2_changed"].sum()),
        })
    by_case = pd.DataFrame(summary)

    transitions = (
        cand.groupby(["case", "v1_scope", "v2_scope"], dropna=False)
        .size().reset_index(name="n_tweets")
        .sort_values(["case", "v1_scope", "v2_scope"])
    )

    assert len(df) == df[["case", "id"]].drop_duplicates().shape[0]
    assert int(by_case["unique_real"].sum()) == len(df)
    assert int(by_case["keyword_candidates"].sum()) == len(cand)
    assert (by_case[[
        "target_current", "target_with_history", "historical_only", "needs_review"
    ]].sum(axis=1) == by_case["keyword_candidates"]).all()

    # Simpan V2 dengan nama baru; JANGAN overwrite V1.
    csv_write_atomic(REPORT_DIR / "event_scope_v2_by_tweet.csv", df)
    csv_write_atomic(REPORT_DIR / "event_scope_v2_by_case.csv", by_case)
    csv_write_atomic(REPORT_DIR / "event_scope_v2_transitions.csv", transitions)

    # Review prioritas: semua ambigu/historis/perubahan dari V1.
    # Tambah kontrol deterministik ~20% dari label yang tidak berubah,
    # untuk memeriksa false positive yang lolos di kedua versi.
    queue_path = REPORT_DIR / "event_scope_v2_manual_review.csv"
    priority_mask = (
        cand["v2_scope"].isin(["NEEDS_REVIEW", "HISTORICAL_ONLY"])
        | cand["v1_to_v2_changed"]
    )
    unchanged_target = cand["v2_scope"].isin([
        "TARGET_CURRENT", "TARGET_WITH_HISTORY"
    ]) & ~cand["v1_to_v2_changed"]
    sample_unchanged = pd.Series([
        int(hashlib.sha256(
            f"{r.case}:{r.id}".encode("utf-8")
        ).hexdigest()[:8], 16) % 5 == 0
        for r in cand.itertuples(index=False)
    ], index=cand.index)
    queue_mask = priority_mask | (unchanged_target & sample_unchanged)
    queue = cand.loc[queue_mask, [
        "case", "id", "created_at_source", "text", "v1_scope", "v2_scope",
        "v2_reason", "v1_to_v2_changed",
    ]].copy()
    queue["review_stratum"] = [
        "priority" if priority_mask.loc[i] else "sample_unchanged"
        for i in queue.index
    ]
    queue["manual_scope"] = ""
    queue["manual_notes"] = ""
    queue["reviewer"] = ""
    queue["reviewed_at"] = ""
    if queue_path.exists():
        prev = pd.read_csv(queue_path, dtype={"id": "string"}, keep_default_na=False)
        if prev.duplicated(["case", "id"]).any():
            raise ValueError("Ada ID ganda dalam antrean review lama.")
        old_ids = set(zip(prev["case"].astype(str), prev["id"].astype(str)))
        fresh = queue.loc[[
            (str(r.case), str(r.id)) not in old_ids
            for r in queue.itertuples(index=False)
        ]]
        # Baris lama tidak diubah agar label dan catatan manual aman.
        queue = pd.concat([prev, fresh], ignore_index=True)
    csv_write_atomic(queue_path, queue)

    display(by_case)
    display(transitions)
    print("ID unik asli:", len(df))
    print("Kandidat keyword:", len(cand))
    print("Relevan otomatis V2:", int(by_case["event_relevant_auto"].sum()))
    print("Berubah dari V1:", int(cand["v1_to_v2_changed"].sum()))
    print("Antrean review (termasuk label yang sudah dikerjakan):", len(queue))
    return df, by_case, transitions, queue


df_scope_v2, df_scope_v2_case, df_scope_v2_transitions, df_scope_v2_review = report_event_scope_v2()


,case,unique_real,keyword_candidates,target_current,target_with_history,historical_only,needs_review,event_relevant_auto,changed_vs_v1
0,lombok_2018,53,52,52,0,0,0,52,0
1,palu_2018,60,57,56,1,0,0,57,0
2,sumatra_2025,119,115,90,4,9,12,94,33


,case,v1_scope,v2_scope,n_tweets
0,lombok_2018,TARGET_CURRENT,TARGET_CURRENT,52
1,palu_2018,TARGET_CURRENT,TARGET_CURRENT,56
2,palu_2018,TARGET_WITH_HISTORY,TARGET_WITH_HISTORY,1
3,sumatra_2025,HISTORICAL_ONLY,HISTORICAL_ONLY,9
4,sumatra_2025,HISTORICAL_ONLY,NEEDS_REVIEW,9
5,sumatra_2025,HISTORICAL_ONLY,TARGET_WITH_HISTORY,2
6,sumatra_2025,NEEDS_REVIEW,NEEDS_REVIEW,3
7,sumatra_2025,NEEDS_REVIEW,TARGET_CURRENT,21
8,sumatra_2025,NEEDS_REVIEW,TARGET_WITH_HISTORY,1
9,sumatra_2025,TARGET_CURRENT,TARGET_CURRENT,69


ID unik asli: 232
Kandidat keyword: 224
Relevan otomatis V2: 203
Berubah dari V1: 33
Antrean review (termasuk label yang sudah dikerjakan): 84


In [ ]:
# 11E — EVALUASI REVIEW MANUAL (OPSIONAL, OFFLINE)
# Isi "manual_scope" terlebih dahulu pada event_scope_v2_manual_review.csv
# dengan salah satu kategori: TARGET_CURRENT, TARGET_WITH_HISTORY,
# HISTORICAL_ONLY, NEEDS_REVIEW.
from IPython.display import display

review_path = REPORT_DIR / "event_scope_v2_manual_review.csv"
review_df = pd.read_csv(review_path, dtype={"id": "string"}, keep_default_na=False)
allowed = {"TARGET_CURRENT", "TARGET_WITH_HISTORY", "HISTORICAL_ONLY", "NEEDS_REVIEW"}
invalid = review_df.loc[
    review_df["manual_scope"].ne("") & ~review_df["manual_scope"].isin(allowed)
]
if len(invalid):
    raise ValueError("Nilai manual_scope tidak valid: " + ", ".join(invalid["manual_scope"].unique()))

labeled = review_df.loc[review_df["manual_scope"].isin(allowed)].copy()
print("Sudah direview:", len(labeled), "/", len(review_df))

if not labeled.empty:
    print("Matriks prediksi V2 (baris) versus keputusan manual (kolom):")
    display(pd.crosstab(
        labeled["v2_scope"], labeled["manual_scope"],
        rownames=["V2"], colnames=["Manual"]
    ))
    print("Persentase kecocokan pada sampel berlabel:",
          round(100 * labeled["v2_scope"].eq(labeled["manual_scope"]).mean(), 1), "%")
    print("Catatan: persentase ini bukan akurasi seluruh dataset (sampel review dipilih secara terarah).")
else:
    print("Belum ada label manual. Ini normal sebelum tahap review.")


In [33]:
# 12 — CONTOH TWEET UNTUK AUDIT SEMANTIK (TANPA API)
def preview_policy_samples(max_each_case=5):
    if max_each_case < 1:
        raise ValueError('max_each_case minimal 1')
    state = load_checkpoint()
    for case in FULL_CASES:
        print('\n' + '=' * 70 + '\nCONTOH:', case)
        shown, seen = 0, set()
        for record in state.values():
            if record.get('status') != 'SAVED' or record.get('case') != case:
                continue
            rel = record.get('raw_path')
            if not rel:
                continue
            path = PROJECT_ROOT / Path(str(rel).replace('\\', '/'))
            if not path.exists():
                continue
            with path.open('r', encoding='utf-8') as f:
                for line in f:
                    item = json.loads(line)
                    if item.get('type') != 'tweet' or not item.get('id'):
                        continue
                    key = str(item['id'])
                    if key in seen:
                        continue
                    seen.add(key)
                    print('-', str(item.get('text', ''))[:350].replace('\n', ' '))
                    shown += 1
                    if shown >= max_each_case:
                        break
            if shown >= max_each_case:
                break
        if not shown:
            print('(Tidak ada tweet tersimpan yang tersedia untuk contoh)')

# Preview ini bukan labeling dan bukan bukti representativitas.


## Keputusan produksi dan catatan metodologis

- Jangan menggunakan nilai `CREDIT_USED_TO_DATE_USD` sebagai saldo yang dapat dibelanjakan. **Saldo aktual dashboard** wajib diisikan pada `AVAILABLE_CREDIT_USD` sebelum menjalankan batch berbayar.
- **Tidak ada panggilan API pada mode preview.** Cell `09B` memakai fake Actor dan direktori sementara. Cost lookup historis default OFF.
- `MAX_CHARGE_PER_RUN_USD` (per run API) ditambah komitmen maksimum setiap run dan `MAX_NEW_RUNS` (per sesi) adalah proteksi berlapis; nilai batas dapat berbeda dari tagihan platform karena pembulatan, peristiwa billing lain, atau biaya akun lain.
- Jika `RUN_MODE='full'`: satu query `E01_P01` per kasus, setiap 6 jam, total **416 task awal**. Tidak otomatis menyapu seluruh 42 grup keyword. Ini fase 1; perlu evaluasi yield dan jangkauan sebelum menambah query lain.
- `POSSIBLE_TRUNCATION` hanya alarm karena `unique_real_ids >= MAX_ITEMS`; tidak membuktikan jumlah tweet yang hilang. Pengujian 3 jam sebelumnya justru menemukan lebih sedikit ID unik dari 6 jam.
- Satu `SAVED` artinya file disimpan, **bukan** data semantik valid atau representatif. Saat raw->clean, hilangkan mock, duplikasi ID, duplikasi konten, promosi/berita yang tidak relevan, dan nilai kecukupan temporal.
- Jangan menyebut persentase sentimen sebagai representasi seluruh masyarakat Indonesia; basisnya adalah post X yang tersedia dalam sampel query.
